In [ ]:
!mkdir timeseries
%cd timeseries
!mkdir config data experiments models utils

/content/timeseries


In [ ]:
!gdown https://drive.google.com/drive/folders/1jJF4eyd6jOhtDdBxsBqZxADycebXLKi8?usp=sharing -O . --folder

Retrieving folder contents
Processing file 1eBBuh0tErhOi8pn8MbZSdZsLheWQlJmF electricity.csv
Processing file 10YIsPPBXP67AIYrRV3L5R1zP7qyuSAcu ETTh1.csv
Processing file 1bc2152E8Kz7zT1RUJdEnvfYnaeNSu3so ETTh2.csv
Processing file 1pM6fCIjuOYs572Rxh17hS-TyyGp1vGjZ ETTm1.csv
Processing file 1CRfjDX4Cgb15Y372j--cnjZQ-ONYMRXE ETTm2.csv
Processing file 1O22tzJo80P4CdhF3s3OWvjehV1wARRtC national_illness.csv
Processing file 1X2ICEGDQRUR0GbiOcCvE80lrDGYRtj3F timeseries.csv
Processing file 1VK13chaIDj8bmSajDfmWGECeZBwx3NEt traffic.csv
Processing file 1dcK4jg9qpeEYH7383s1YH6RXKK3cFzBX weather.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1eBBuh0tErhOi8pn8MbZSdZsLheWQlJmF
To: /content/timeseries/electricity.csv
100% 36.4M/36.4M [00:00<00:00, 47.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=10YIsPPBXP67AIYrRV3L5R1zP7qyuSAcu
To: /content/timeseries/ETTh1.csv
100% 2.59M/2.59

#create files

In [ ]:
#@title evaluate.py

%%writefile evaluate.py
#!/usr/bin/env python3
"""
Evaluation script for trained models.
"""

import torch
import yaml
import argparse
import os
from torch.utils.data import DataLoader
import joblib

from data.preprocessing import TimeSeriesPreprocessor
from data.dataset import TimeSeriesDataset
from models.llm_model import EnhancedTimeSeriesLLM
from utils.metrics import evaluate_model, print_metrics
from utils.helpers import (
    load_checkpoint, plot_predictions, plot_error_distribution,
    save_json, get_device
)


def parse_arguments():
    """Parse command line arguments."""
    parser = argparse.ArgumentParser(
        description='Evaluate trained Time Series LLM'
    )

    parser.add_argument(
        '--config',
        type=str,
        default='config/config.yaml',
        help='Path to configuration file'
    )

    parser.add_argument(
        '--dataset',
        type=str,
        required=True,
        help='Dataset name to evaluate (e.g., timeseries, ETTh1)'
    )

    parser.add_argument(
        '--checkpoint',
        type=str,
        default=None,
        help='Path to model checkpoint. If not provided, uses best_model_<dataset>.pth'
    )

    parser.add_argument(
        '--split',
        type=str,
        default='test',
        choices=['train', 'val', 'test'],
        help='Which split to evaluate on'
    )

    parser.add_argument(
        '--batch_size',
        type=int,
        default=32,
        help='Batch size for evaluation'
    )

    parser.add_argument(
        '--save_plots',
        action='store_true',
        help='Save prediction plots'
    )

    parser.add_argument(
        '--output_dir',
        type=str,
        default='evaluation_results',
        help='Directory to save evaluation results'
    )

    return parser.parse_args()


def evaluate_model_on_dataset(model, dataloader, device, scaler,
                              save_plots=False, output_dir=None, dataset_name=None):
    """
    Evaluate model and optionally save plots.
    """
    # Evaluate
    metrics, y_pred, y_true = evaluate_model(model, dataloader, device, scaler)

    # Print metrics
    print_metrics(metrics, prefix=f"{dataset_name} - ")

    # Save plots if requested
    if save_plots and output_dir:
        os.makedirs(output_dir, exist_ok=True)

        # Predictions plot
        plot_predictions(
            y_true, y_pred,
            save_path=os.path.join(output_dir, f'{dataset_name}_predictions.png'),
            title=f'{dataset_name} - Predictions vs Ground Truth'
        )

        # Error distribution plot
        plot_error_distribution(
            y_true, y_pred,
            save_path=os.path.join(output_dir, f'{dataset_name}_errors.png'),
            title=f'{dataset_name} - Error Distribution'
        )

    return metrics, y_pred, y_true


def main():
    """Main evaluation function."""
    args = parse_arguments()

    # Load config
    with open(args.config, 'r') as f:
        config = yaml.safe_load(f)

    # Get device
    device = get_device(config['device'])
    print(f"Using device: {device}\n")

    # Find dataset config
    dataset_config = None
    for ds in config['data']['datasets']:
        if ds['name'] == args.dataset:
            dataset_config = ds
            break

    if dataset_config is None:
        print(f"Error: Dataset '{args.dataset}' not found in config!")
        return

    # Load and prepare data
    print(f"Loading dataset: {args.dataset}")
    preprocessor = TimeSeriesPreprocessor(config)
    data_dict = preprocessor.prepare_dataset(dataset_config, save_scaler=False)

    # Select split
    if args.split == 'train':
        X = data_dict['X_train']
        y = data_dict['y_train']
    elif args.split == 'val':
        X = data_dict['X_val']
        y = data_dict['y_val']
    else:  # test
        X = data_dict['X_test']
        y = data_dict['y_test']

    print(f"Evaluating on {args.split} split: {X.shape[0]} samples\n")

    # Create dataloader
    d_model = config['model']['llm']['d_model']
    dataset = TimeSeriesDataset(X, y, d_model, augmentation=False)
    dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False)

    # Initialize model
    model = EnhancedTimeSeriesLLM(
        input_dim=data_dict['input_dim'],
        output_dim=data_dict['target_dim'],
        **config['model']['llm']
    ).to(device)

    # Load checkpoint
    if args.checkpoint is None:
        args.checkpoint = os.path.join(
            config['paths']['model_dir'],
            f'best_model_{args.dataset}.pth'
        )

    if not os.path.exists(args.checkpoint):
        print(f"Error: Checkpoint not found at {args.checkpoint}")
        return

    print(f"Loading checkpoint: {args.checkpoint}")
    checkpoint = load_checkpoint(args.checkpoint, model, device=device)
    print(f"Loaded model from epoch {checkpoint['epoch']}")
    print(f"Training loss: {checkpoint['loss']:.6f}\n")

    # Load scaler
    scaler_path = os.path.join(
        config['paths']['model_dir'],
        f'scaler_{args.dataset}.pkl'
    )

    if os.path.exists(scaler_path):
        scaler = joblib.load(scaler_path)
    else:
        print(f"Warning: Scaler not found at {scaler_path}")
        print("Results will be in normalized space.")
        scaler = None

    # Evaluate
    print("="*80)
    print(f"Evaluating on {args.split} split")
    print("="*80 + "\n")

    metrics, y_pred, y_true = evaluate_model_on_dataset(
        model, dataloader, device, scaler,
        save_plots=args.save_plots,
        output_dir=args.output_dir,
        dataset_name=f"{args.dataset}_{args.split}"
    )

    # Save metrics
    os.makedirs(args.output_dir, exist_ok=True)
    results = {
        'dataset': args.dataset,
        'split': args.split,
        'checkpoint': args.checkpoint,
        'metrics': metrics,
        'config': config
    }

    save_json(
        results,
        os.path.join(args.output_dir, f'eval_{args.dataset}_{args.split}.json')
    )

    print(f"\nResults saved to: {args.output_dir}")


if __name__ == '__main__':
    main()

Overwriting evaluate.py


In [ ]:
#@title main.py
%%writefile main.py
#!/usr/bin/env python3
import torch
import yaml
import argparse
import os
import sys
import gc
import numpy as np
from pprint import pprint

from data.preprocessing import TimeSeriesPreprocessor
from experiments.train import train_on_dataset

# BUG 1 FIXED: removed stray module-level HORIZONS/DATASETS benchmark loop
# that referenced `config`, `data_dict`, `device` before they were defined.
# Those variables only exist inside main() after the config is loaded.

MAX_INPUT_DIM = 200


def parse_arguments():
    parser = argparse.ArgumentParser(description='Train Time Series LLM')
    parser.add_argument('--config', type=str, default='config/config.yaml')
    parser.add_argument('--datasets', nargs='+', default=None,
        help='Datasets to train on. If omitted, trains all in config.')
    parser.add_argument('--device', type=str, default=None)
    parser.add_argument('--epochs', type=int, default=None)
    parser.add_argument('--batch_size', type=int, default=None)
    parser.add_argument('--lr', type=float, default=None)
    parser.add_argument('--max_samples', type=int, default=None,
        help='Max train samples per dataset. Omit for no limit.')
    parser.add_argument('--max_input_dim', type=int, default=MAX_INPUT_DIM,
        help='Skip datasets whose input_dim exceeds this.')
    return parser.parse_args()


def main():
    args = parse_arguments()

    # ── T4 GPU setup ──────────────────────────────────────────────────────
    torch.backends.cudnn.benchmark        = True
    torch.backends.cudnn.deterministic    = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    # ──────────────────────────────────────────────────────────────────────

    if not os.path.exists(args.config):
        raise FileNotFoundError(f"Config file not found: {args.config}")
    with open(args.config, 'r') as f:
        config = yaml.safe_load(f)

    # Apply CLI overrides
    if args.device is not None:     config['device'] = args.device
    if args.epochs is not None:     config['training']['epochs'] = args.epochs
    if args.batch_size is not None: config['training']['batch_size'] = args.batch_size
    if args.lr is not None:         config['training']['learning_rate'] = args.lr

    print("\n" + "="*80)
    print("Configuration:")
    print("="*80)
    pprint(config)
    print("="*80 + "\n")

    for key in ['model_dir', 'results_dir', 'log_dir']:
        os.makedirs(config['paths'][key], exist_ok=True)

    device_str = config['device']
    if device_str == 'cuda' and not torch.cuda.is_available():
        print("WARNING: CUDA not available, falling back to CPU")
        device_str = 'cpu'
    device = torch.device(device_str)
    print(f"Using device: {device}\n")

    dataset_configs = config['data']['datasets']
    if args.datasets:
        dataset_configs = [d for d in dataset_configs if d['name'] in args.datasets]
    if not dataset_configs:
        print("ERROR: No datasets matched.")
        sys.exit(1)

    print(f"Training on {len(dataset_configs)} dataset(s): "
          f"{[d['name'] for d in dataset_configs]}\n")

    preprocessor = TimeSeriesPreprocessor(config)
    all_results = {}

    for dataset_config in dataset_configs:
        dataset_name = dataset_config['name']
        print(f"\n{'#'*80}")
        print(f"# Training on: {dataset_name}")
        print(f"{'#'*80}\n")

        try:
            data_dict = preprocessor.prepare_dataset(dataset_config)

            if data_dict['input_dim'] > args.max_input_dim:
                print(f"Skipping {dataset_name}: input_dim={data_dict['input_dim']} "
                      f"> limit {args.max_input_dim}")
                del data_dict
                continue

            n_train = len(data_dict['X_train'])
            if args.max_samples is not None and n_train > args.max_samples:
                print(f"Subsampling {dataset_name}: {n_train} → {args.max_samples} samples")
                idx = np.random.choice(n_train, args.max_samples, replace=False)
                data_dict['X_train'] = data_dict['X_train'][idx]
                data_dict['y_train'] = data_dict['y_train'][idx]

            results = train_on_dataset(dataset_name, data_dict, config, device)
            all_results[dataset_name] = results

            m = results['test_metrics']
            print(f"\n✓ {dataset_name} — MSE: {m['MSE']:.6f} | MAE: {m['MAE']:.6f} | "
                  f"HR: {m['HR']:.2f}% | TCE: {m['TCE']:.6f} | SDTW: {m['Soft-DTW']:.6f}")

        except Exception as e:
            print(f"\n✗ Error on {dataset_name}: {e}")
            import traceback; traceback.print_exc()
        finally:
            if 'data_dict' in dir():
                del data_dict
            gc.collect()
            torch.cuda.empty_cache()

    print("\n" + "="*80)
    print("Training Summary")
    print("="*80)
    if all_results:
        for name, res in all_results.items():
            m = res['test_metrics']
            print(f"\n{name}:")
            print(f"  MSE={m['MSE']:.6f}  MAE={m['MAE']:.6f}  RMSE={m['RMSE']:.6f}")
            print(f"  HR={m['HR']:.2f}%  TCE={m['TCE']:.6f}  Soft-DTW={m['Soft-DTW']:.6f}")
    else:
        print("No datasets trained successfully.")
    print("="*80 + "\n")


if __name__ == '__main__':
    main()

Overwriting main.py


In [ ]:
#@title requirements.txt
%%writefile requirements.txt
torch>=2.0.0
numpy>=1.24.0
pandas>=2.0.0
scikit-learn>=1.3.0
matplotlib>=3.7.0
seaborn>=0.12.0
joblib>=1.3.0
pyyaml>=6.0
tqdm>=4.65.0
tensorboard>=2.13.0
sentence-transformers>=2.2.0
transformers>=4.35.0

Overwriting requirements.txt


In [ ]:
#@title ablation.py
%%writefile ablation.py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import os
import sys
import gc
import json
import yaml
import argparse
from tqdm import tqdm
from typing import Dict, List
import pandas as pd

from models.llm_model import EnhancedTimeSeriesLLM, AblationLLM
from models.losses import CombinedLoss
from data.dataset import AblationDataset
from data.preprocessing import TimeSeriesPreprocessor
from utils.metrics import evaluate_model, print_metrics
from experiments.train import Trainer


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating):  return float(obj)
        if isinstance(obj, np.integer):   return int(obj)
        if isinstance(obj, np.ndarray):   return obj.tolist()
        return super().default(obj)


class AblationExperiment:

    def __init__(self, config: Dict, device: torch.device):
        self.config = config
        self.device = device

    def run_ablation(self, ablation_config: Dict, data_dict: Dict,
                     dataset_name: str) -> Dict:

        ablation_name      = ablation_config['name']
        disabled_components = ablation_config['disable']

        print(f"\n{'='*60}")
        print(f"Running Ablation: {ablation_name}")
        print(f"Disabled: {disabled_components}  |  Dataset: {dataset_name}")
        print(f"{'='*60}\n")

        d_model    = self.config['model']['llm']['d_model']
        batch_size = self.config['training']['batch_size']

        train_dataset = AblationDataset(data_dict['X_train'], data_dict['y_train'],
                                        d_model, ablation_config)
        val_dataset   = AblationDataset(data_dict['X_val'],   data_dict['y_val'],
                                        d_model, ablation_config)
        test_dataset  = AblationDataset(data_dict['X_test'],  data_dict['y_test'],
                                        d_model, ablation_config)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                                  pin_memory=True, num_workers=2, prefetch_factor=4,
                                  persistent_workers=True)

        val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                                  pin_memory=True, num_workers=4, prefetch_factor=2)
        test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                                  pin_memory=True, num_workers=4, prefetch_factor=2)

        prediction_length = self.config['data'].get('prediction_length', 1)

        base_model = EnhancedTimeSeriesLLM(
            input_dim=data_dict['input_dim'],
            output_dim=data_dict['target_dim'],
            prediction_length=prediction_length,
            **self.config['model']['llm']
        )
        model = AblationLLM(base_model, ablation_config)

        # Adjust loss weights based on ablation
        loss_config = self.config['loss'].copy()
        if 'tcca'        in disabled_components: loss_config['lambda_scta']       = 0.0
        if 'cross_modal' in disabled_components: loss_config['lambda_crossmodal'] = 0.0

        ablation_config_copy = self.config.copy()
        ablation_config_copy['loss'] = loss_config

        trainer = Trainer(model, ablation_config_copy, self.device)

        checkpoint_name = f"{dataset_name}_{ablation_name}"
        training_history = trainer.train(train_loader, val_loader, checkpoint_name)

        trainer.load_checkpoint(checkpoint_name)
        test_metrics, y_pred, y_true = evaluate_model(
            trainer.model, test_loader, self.device, data_dict['scaler']
        )
        print_metrics(test_metrics, prefix=f"{ablation_name} - ")

        return {
            'ablation_name':      ablation_name,
            'disabled_components': disabled_components,
            'test_metrics':       test_metrics,
            'training_history':   training_history
        }

    def run_all_ablations(self, data_dict: Dict, dataset_name: str,
                          filter_ablations: List[str] = None) -> Dict:

        experiments = self.config['ablation']['experiments']
        if filter_ablations:
            experiments = [e for e in experiments if e['name'] in filter_ablations]

        results = {}
        for ablation_config in experiments:
            name = ablation_config['name']
            try:
                results[name] = self.run_ablation(ablation_config, data_dict, dataset_name)
            except Exception as e:
                print(f"\nError in ablation {name}: {e}")
                import traceback; traceback.print_exc()

        return results

    def compare_ablations(self, results: Dict, dataset_name: str):
        if not results:
            print(f"No successful ablations for {dataset_name}, skipping comparison.")
            return pd.DataFrame()

        rows = []
        for name, result in results.items():
            m = result['test_metrics']
            rows.append({
                'Ablation':   name,
                'MSE':        m['MSE'],
                'MAE':        m['MAE'],
                'RMSE':       m['RMSE'],
                'HR (%)':     m['HR'],
                'Perplexity': m['Perplexity'],
                'TCE':        m['TCE'],
                'Soft-DTW':   m['Soft-DTW'],
            })

        df = pd.DataFrame(rows).sort_values('MSE')

        print(f"\n{'='*80}")
        print(f"Ablation Results — {dataset_name}")
        print(f"{'='*80}")
        print(df.to_string(index=False))
        print(f"{'='*80}\n")

        results_dir = self.config['paths']['results_dir']
        os.makedirs(results_dir, exist_ok=True)
        csv_path = os.path.join(results_dir, f'ablation_comparison_{dataset_name}.csv')
        df.to_csv(csv_path, index=False)
        print(f"Saved comparison to: {csv_path}")
        return df

    def save_all_results(self, all_results: Dict):
        results_dir = self.config['paths']['results_dir']
        os.makedirs(results_dir, exist_ok=True)
        path = os.path.join(results_dir, 'ablation_results_all.json')
        with open(path, 'w') as f:
            json.dump(all_results, f, indent=2, cls=NumpyEncoder)
        print(f"\nSaved all results to: {path}")


def parse_arguments():
    parser = argparse.ArgumentParser(description='Ablation study for Time Series LLM')

    parser.add_argument('--config', type=str, default='config/config.yaml')
    parser.add_argument('--datasets', nargs='+', default=None,
        help='Datasets to ablate on. If omitted, uses all in config.')
    parser.add_argument('--ablations', nargs='+', default=None,
        help='Specific ablations to run (e.g., full_model w/o_TCCA).')
    parser.add_argument('--device', type=str, default=None)
    parser.add_argument('--epochs', type=int, default=None)
    parser.add_argument('--batch_size', type=int, default=None)
    parser.add_argument('--lr', type=float, default=None)
    parser.add_argument('--max_samples', type=int, default=5000,
        help='Max training samples per dataset.')
    parser.add_argument('--max_input_dim', type=int, default=200,
        help='Skip datasets with more input features than this.')

    return parser.parse_args()


def main():
    args = parse_arguments()

    # Load config FIRST
    if not os.path.exists(args.config):
        raise FileNotFoundError(f"Config not found: {args.config}")
    with open(args.config, 'r') as f:
        config = yaml.safe_load(f)

    # Apply CLI overrides AFTER loading config
    if args.device is not None:     config['device'] = args.device
    if args.epochs is not None:     config['training']['epochs'] = args.epochs
    if args.batch_size is not None: config['training']['batch_size'] = args.batch_size
    if args.lr is not None:         config['training']['learning_rate'] = args.lr

    # Device
    device_str = config['device']
    if device_str == 'cuda' and not torch.cuda.is_available():
        print("WARNING: CUDA unavailable, using CPU")
        device_str = 'cpu'
    device = torch.device(device_str)
    print(f"Using device: {device}")

    # Dataset configs
    dataset_configs = config['data']['datasets']
    if args.datasets:
        dataset_configs = [d for d in dataset_configs if d['name'] in args.datasets]
    if not dataset_configs:
        print("ERROR: No datasets matched.")
        sys.exit(1)

    print(f"\nRunning ablations on {len(dataset_configs)} dataset(s): "
          f"{[d['name'] for d in dataset_configs]}\n")

    preprocessor   = TimeSeriesPreprocessor(config)
    ablation_exp   = AblationExperiment(config, device)
    all_results    = {}

    for dataset_config in dataset_configs:
        dataset_name = dataset_config['name']
        print(f"\n{'#'*80}")
        print(f"# Ablations on dataset: {dataset_name}")
        print(f"{'#'*80}\n")

        try:
            data_dict = preprocessor.prepare_dataset(dataset_config)

            # Skip high-dimensional datasets
            if data_dict['input_dim'] > args.max_input_dim:
                print(f"Skipping {dataset_name}: input_dim={data_dict['input_dim']} "
                      f"> limit {args.max_input_dim}")
                del data_dict
                continue

            # Subsample
            n_train = len(data_dict['X_train'])
            if n_train > args.max_samples:
                print(f"Subsampling: {n_train} → {args.max_samples}")
                idx = np.random.choice(n_train, args.max_samples, replace=False)
                data_dict['X_train'] = data_dict['X_train'][idx]
                data_dict['y_train'] = data_dict['y_train'][idx]

            results = ablation_exp.run_all_ablations(
                data_dict, dataset_name, filter_ablations=args.ablations
            )
            all_results[dataset_name] = results
            ablation_exp.compare_ablations(results, dataset_name)

        except Exception as e:
            print(f"\n✗ Error on {dataset_name}: {e}")
            import traceback; traceback.print_exc()
        finally:
            if 'data_dict' in dir():
                del data_dict
            gc.collect()
            torch.cuda.empty_cache()

    ablation_exp.save_all_results(all_results)
    print("\n" + "="*80)
    print("Ablation study completed!")
    print("="*80)


if __name__ == '__main__':
    main()

Overwriting ablation.py


In [ ]:
#@title config.py
%%writefile config/config.yaml
# Configuration for Time Series LLM Project
# BUG 8 FIXED: `data:` was defined twice. The second block
# (seq_length / prediction_length) overwrote the first (datasets list + splits),
# making config['data']['datasets'] disappear at runtime.
# Both blocks are now merged into a single `data:` section.

data:
  datasets:
    - name: "timeseries"
      path: "timeseries.csv"
      timestamp_col: "timestamp"
      timestamp_format: "%d/%m/%Y"
      target_cols: ["target_WS", "target_RH", "target_T"]
      feature_cols: null

    - name: "ETTh1"
      path: "ETTh1.csv"
      timestamp_col: "date"
      timestamp_format: "%Y-%m-%d %H:%M:%S"
      target_cols: ["OT"]
      feature_cols: ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL"]

    - name: "ETTh2"
      path: "ETTh2.csv"
      timestamp_col: "date"
      timestamp_format: "%Y-%m-%d %H:%M:%S"
      target_cols: ["OT"]
      feature_cols: ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL"]

    - name: "ETTm1"
      path: "ETTm1.csv"
      timestamp_col: "date"
      timestamp_format: "%Y-%m-%d %H:%M:%S"
      target_cols: ["OT"]
      feature_cols: ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL"]

    - name: "ETTm2"
      path: "ETTm2.csv"
      timestamp_col: "date"
      timestamp_format: "%Y-%m-%d %H:%M:%S"
      target_cols: ["OT"]
      feature_cols: ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL"]

    - name: "traffic"
      path: "traffic.csv"
      timestamp_col: "date"
      timestamp_format: "%d/%m/%Y %H:%M"
      target_cols: ["OT"]
      feature_cols: null

    - name: "electricity"
      path: "electricity.csv"
      timestamp_col: "date"
      timestamp_format: "%d/%m/%Y %H:%M"
      target_cols: ["OT"]
      feature_cols: null

    - name: "weather"
      path: "weather.csv"
      timestamp_col: "date"
      timestamp_format: "%d/%m/%Y %H:%M"
      target_cols: ["OT"]
      feature_cols: null

    - name: "national_illness"
      path: "national_illness.csv"
      timestamp_col: "date"
      timestamp_format: "%d/%m/%Y %H:%M"
      target_cols: ["OT"]
      feature_cols: null

  # Train/Val/Test split ratios
  test_size: 100
  val_size: 30

  # Sequence parameters (previously in a separate duplicate `data:` block)
  seq_length: 96
  prediction_length: 96    # Standard benchmark; use 96/192/336/720 for full table


model:
  hsqp:
    patch_length:  24
    stride:        12
    tol:           0.01
    alpha:         0.01
    k:             26
    bit_width:     8
    quant_method:  "affine"
    embedding_dim: 64

  llm:
    d_model:       512
    n_heads:       8
    n_layers:      8
    d_ff:          2048
    dropout:       0.1
    proj_out_dim:  128
    revin_affine:  true
    use_hsqp:      true
    patch_len:     24
    patch_stride:  12

loss:
  lambda_forecast:   1.0   # unchanged
  lambda_scta:       0.1   # was 0.3 — SCTA absolute magnitude is ~5x forecast
  lambda_spectral:   0.02  # was 0.05 — spectral magnitude is ~20x forecast
  lambda_crossmodal: 0.5   # was 1.0
  lambda_softcl:     0.5   # was 1.0
  temperature:       0.07  # unchanged

training:
  epochs:       50     # was 20 — PatchTST trains 100, TimeLLM 10–30
  batch_size:   1024    # was 192 — T4 has 15GB, model uses ~180MB, room to grow
  learning_rate: 0.0001
  patience:     40 #12     # keep — still functional with 50 max epochs
  min_delta:    0.0001
  accum_steps:  2

ablation:
  experiments:
    - name: "full_model"
      disable: []
    - name: "w/o_TCCA"
      disable: ["tcca"]
    - name: "w/o_Soft-DTW"
      disable: ["soft_dtw"]
    - name: "w/o_Cross-Modal"
      disable: ["cross_modal"]
    - name: "w/o_SQT"
      disable: ["sqt"]
    - name: "w/o_Quantization"
      disable: ["quantization"]


device: "cuda"

paths:
  data_dir:    "data/"
  model_dir:   "saved_models/"
  results_dir: "results/"
  log_dir:     "logs/"

Overwriting config/config.yaml


In [ ]:
#@title dataset.py
%%writefile data/dataset.py
"""
dataset.py
"""
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import numpy as np
from typing import Dict, Tuple, Optional
from sentence_transformers import SentenceTransformer


# ---------------------------------------------------------------------------
# Sentence-transformer helpers
# ---------------------------------------------------------------------------

def _generate_ts_description(x: np.ndarray, dataset_name: str = "") -> str:
    ch = x[:, 0] if x.ndim == 2 else x
    mean_v    = float(np.mean(ch))
    std_v     = float(np.std(ch))
    trend     = 'increasing' if ch[-1] > ch[0] else 'decreasing'
    slope     = float((ch[-1] - ch[0]) / max(len(ch), 1))
    acf1      = float(np.corrcoef(ch[:-1], ch[1:])[0, 1]) if len(ch) > 2 else 0.0
    autocorr  = 'high' if abs(acf1) > 0.7 else 'moderate' if abs(acf1) > 0.3 else 'low'
    volatility = 'high' if std_v > abs(mean_v) * 0.3 else 'low'
    return (
        f"Dataset: {dataset_name}. "
        f"Time series window: {trend} trend (slope={slope:.4f}). "
        f"Mean={mean_v:.3f}, std={std_v:.3f}. "
        f"Autocorrelation: {autocorr} (lag-1={acf1:.3f}). "
        f"Volatility: {volatility}."
    )

_ST_MODEL = None
_ST_DIM   = 384

def _get_st_model():
    global _ST_MODEL
    if _ST_MODEL is None:
        _ST_MODEL = SentenceTransformer('all-MiniLM-L6-v2')
        _ST_MODEL.eval()
    return _ST_MODEL


def _build_st_embeddings(X: np.ndarray, d_model: int,
                          dataset_name: str = "", seed: int = 42,
                          max_encode: int = 5000) -> torch.Tensor:
    """
    Pre-compute sentence-transformer embeddings.
    Encodes up to max_encode samples; for larger datasets the embeddings
    are tiled/cycled so every sample still gets a unique-enough vector
    without spending minutes encoding 70k descriptions.
    Returns a [N, d_model] tensor (detached, no grad, always on CPU).
    """
    import warnings
    N = len(X)
    torch.manual_seed(seed)
    emb_proj = nn.Linear(_ST_DIM, d_model)
    nn.init.xavier_uniform_(emb_proj.weight)

    n_encode = min(N, max_encode)
    model = _get_st_model()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        descriptions = [
            _generate_ts_description(X[i], dataset_name=dataset_name)
            for i in range(n_encode)
        ]

    with torch.no_grad():
        raw_embs = model.encode(
            descriptions, batch_size=256,
            convert_to_tensor=True, show_progress_bar=True
        )  # [n_encode, 384]
        # Always keep on CPU — emb_proj lives on CPU
        base_emb = emb_proj(raw_embs.float().cpu()).detach()  # [n_encode, d_model]

    # Tile to cover all N samples if N > max_encode
    if N > n_encode:
        repeats = (N + n_encode - 1) // n_encode
        T_emb = base_emb.repeat(repeats, 1)[:N]
    else:
        T_emb = base_emb

    print(f"  ✓ T_emb shape: {T_emb.shape}, "
          f"norm ≈ {T_emb.norm(dim=1).mean():.3f}")
    return T_emb


class TimeSeriesDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray,
                 d_model: int, augmentation: bool = True,
                 dataset_name: str = "", embedding_seed: int = 42):
        self.X            = torch.FloatTensor(X)
        self.y            = torch.FloatTensor(y)
        self.d_model      = d_model
        self.augmentation = augmentation

        print(f"Pre-computing text embeddings for {len(X)} samples…")
        self.T_emb = _build_st_embeddings(X, d_model, dataset_name, embedding_seed)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X_i = self.X[idx]
        X_j = X_i + torch.randn_like(X_i) * 0.01 if self.augmentation else X_i.clone()
        T_emb = self.T_emb[idx]
        temporal_weight = torch.tensor(1.0)
        return X_i, X_j, T_emb, temporal_weight, self.y[idx]


# ---------------------------------------------------------------------------
# Ablation dataset
# BUG 5 FIXED: was calling _build_frozen_embeddings() which no longer exists
# (it was commented out). Now calls _build_st_embeddings() for the normal path
# and pre-allocates zeros for the cross_modal ablation path.
# ---------------------------------------------------------------------------

class AblationDataset(Dataset):
    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        d_model: int,
        ablation_config: Dict,
        embedding_seed: int = 42,
    ):
        self.X               = torch.FloatTensor(X)
        self.y               = torch.FloatTensor(y)
        self.d_model         = d_model
        self.ablation_config = ablation_config
        self.disabled        = ablation_config.get('disable', [])

        # BUG 5 FIX: _build_frozen_embeddings was deleted; replaced with
        # _build_st_embeddings for the active path, zeros for ablated path.
        if 'cross_modal' not in self.disabled:
            dataset_name = ablation_config.get('dataset_name', '')
            print(f"  Building ST embeddings for ablation dataset…")
            self.T_emb = _build_st_embeddings(X, d_model, dataset_name, embedding_seed)
        else:
            # Pre-allocate zeros — no compute needed for zeroed modality
            self.T_emb = torch.zeros(len(X), d_model)
            print("  cross_modal ablation active — embeddings set to zeros.")

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, ...]:
        X_i = self.X[idx]

        if 'soft_dtw' in self.disabled:
            X_j             = X_i.clone()
            temporal_weight = torch.tensor(0.0)
        else:
            X_j             = X_i + torch.randn_like(X_i) * 0.01
            temporal_weight = torch.tensor(1.0)

        T_emb = self.T_emb[idx]
        return X_i, X_j, T_emb, temporal_weight, self.y[idx]


Overwriting data/dataset.py


In [ ]:
#@title preprocessing.py
%%writefile data/preprocessing.py
"""
preprocessing.py - Fixed version.
Changes vs original:
  1. Split data BEFORE fitting any scaler (eliminates data leakage).
  2. Replaced MinMaxScaler with StandardScaler on targets (matches baselines,
     preserves variance for attention mechanisms).
  3. Feature scaler is also StandardScaler, fit only on train split.
  4. Scaler is fit on raw numpy arrays from the train split, not the full df.
"""
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler          # ← was MinMaxScaler
import joblib
from typing import Tuple, Dict, Optional, List
import os


class TimeSeriesPreprocessor:
    """Handles data cleaning, feature engineering, and normalization."""

    def __init__(self, config: Dict):
        self.config = config
        self.scalers = {}          # target scalers, keyed by dataset name
        self.feature_scalers = {}  # feature scalers, keyed by dataset name

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def load_and_clean(self, dataset_config: Dict) -> pd.DataFrame:
        try:
            df = pd.read_csv(dataset_config['path'])
        except FileNotFoundError:
            raise FileNotFoundError(f"Dataset not found: {dataset_config['path']}")

        timestamp_col = dataset_config['timestamp_col']
        timestamp_format = dataset_config['timestamp_format']
        df[timestamp_col] = pd.to_datetime(
            df[timestamp_col], format=timestamp_format, errors='coerce'
        )
        df = df.dropna(subset=[timestamp_col])
        df = df.sort_values(timestamp_col).reset_index(drop=True)

        target_cols = dataset_config['target_cols']
        feature_cols = dataset_config.get('feature_cols') or []
        for col in target_cols + feature_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                df[col] = df[col].fillna(df[col].mean())

        print(f"Loaded dataset: {dataset_config['name']}, shape: {df.shape}")
        return df

    def create_temporal_features(self, df: pd.DataFrame, timestamp_col: str) -> pd.DataFrame:
        df['year']      = df[timestamp_col].dt.year
        df['month']     = df[timestamp_col].dt.month
        df['day']       = df[timestamp_col].dt.day
        df['dayofweek'] = df[timestamp_col].dt.dayofweek
        df['hour']      = df[timestamp_col].dt.hour

        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['day_sin']   = np.sin(2 * np.pi * df['day'] / 31)
        df['day_cos']   = np.cos(2 * np.pi * df['day'] / 31)
        return df

    def create_sequences(
        self,
        data: np.ndarray,
        seq_length: int,
        target_dim: int,
        prediction_length: int = 1,
    ) -> Tuple[np.ndarray, np.ndarray]:
        X, y = [], []
        for i in range(len(data) - seq_length - prediction_length + 1):
            X.append(data[i : i + seq_length])
            y.append(data[i + seq_length : i + seq_length + prediction_length, -target_dim:])

        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.float32)
        if prediction_length == 1:
            y = y.squeeze(1)
        return X, y

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def prepare_dataset(
        self, dataset_config: Dict, save_scaler: bool = True
    ) -> Dict[str, np.ndarray]:
        """Complete preprocessing pipeline — leakage-free."""

        # ── 1. Load & clean ───────────────────────────────────────────
        df = self.load_and_clean(dataset_config)
        timestamp_col = dataset_config['timestamp_col']
        df = self.create_temporal_features(df, timestamp_col)

        target_cols  = dataset_config['target_cols']
        feature_cols = dataset_config.get('feature_cols')

        if feature_cols is None:
            feature_cols = [
                col for col in df.select_dtypes(include=[np.number]).columns
                if col not in target_cols
            ]
        else:
            feature_cols = [col for col in feature_cols if col in df.columns]

        temporal_features = ['month_sin', 'month_cos', 'day_sin', 'day_cos', 'dayofweek', 'hour']
        for tf in temporal_features:
            if tf not in feature_cols and tf in df.columns:
                feature_cols.append(tf)

        # Ensure numeric, fill any remaining NaNs
        all_cols = feature_cols + target_cols
        for col in all_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].isna().any():
                    print(f"Warning: NaN values in '{col}', filling with column mean")
                    df[col] = df[col].fillna(df[col].mean())

        # ── 2. Build raw sequences BEFORE any scaling ─────────────────
        # This is the key ordering fix: split first, scale second.
        raw_data = df[feature_cols + target_cols].values.astype(np.float32)

        seq_length        = self.config['data']['seq_length']
        prediction_length = self.config['data'].get('prediction_length', 1)

        X, y = self.create_sequences(raw_data, seq_length, len(target_cols), prediction_length)

        # ── 3. Train / val / test split (on sequences) ────────────────
        test_size = self.config['data']['test_size']
        val_size  = self.config['data']['val_size']

        X_train = X[: -test_size - val_size]
        y_train = y[: -test_size - val_size]

        X_val   = X[-test_size - val_size : -test_size]
        y_val   = y[-test_size - val_size : -test_size]

        X_test  = X[-test_size:]
        y_test  = y[-test_size:]

        # ── 4. Fit scalers on TRAIN split only ────────────────────────
        # Features: columns 0 … n_features-1
        # Targets:  columns -n_targets … -1  (last n_targets cols in X)
        n_features = len(feature_cols)
        n_targets  = len(target_cols)
        dataset_name = dataset_config['name']

        # --- Feature scaler ---
        feature_scaler = StandardScaler()
        # Reshape to 2-D (samples*timesteps, features), fit, reshape back
        train_feat_2d  = X_train[:, :, :n_features].reshape(-1, n_features)
        feature_scaler.fit(train_feat_2d)

        def scale_features(arr):
            n, t, f = arr.shape
            return feature_scaler.transform(arr.reshape(-1, f)).reshape(n, t, f)

        X_train[:, :, :n_features] = scale_features(X_train[:, :, :n_features])
        X_val  [:, :, :n_features] = scale_features(X_val  [:, :, :n_features])
        X_test [:, :, :n_features] = scale_features(X_test [:, :, :n_features])

        # --- Target scaler (StandardScaler, fit on train targets only) ---
        target_scaler = StandardScaler()         # ← replaces MinMaxScaler
        train_tgt_2d  = y_train.reshape(-1, n_targets)
        target_scaler.fit(train_tgt_2d)

        def scale_targets(arr):
            orig_shape = arr.shape
            return target_scaler.transform(arr.reshape(-1, n_targets)).reshape(orig_shape)

        y_train = scale_targets(y_train)
        y_val   = scale_targets(y_val)
        y_test  = scale_targets(y_test)

        # Also scale the target columns inside X (last n_targets channels)
        def scale_x_targets(arr):
            n, t, _ = arr.shape
            tgt_slice = arr[:, :, n_features:].reshape(-1, n_targets)
            scaled    = target_scaler.transform(tgt_slice)
            arr[:, :, n_features:] = scaled.reshape(n, t, n_targets)
            return arr

        X_train = scale_x_targets(X_train)
        X_val   = scale_x_targets(X_val)
        X_test  = scale_x_targets(X_test)

        self.scalers[dataset_name]         = target_scaler
        self.feature_scalers[dataset_name] = feature_scaler

        # ── 5. Sanity checks ──────────────────────────────────────────
        print(f"\n{dataset_name} - Data shapes:")
        print(f"  Train: X={X_train.shape}, y={y_train.shape}")
        print(f"  Val:   X={X_val.shape},   y={y_val.shape}")
        print(f"  Test:  X={X_test.shape},  y={y_test.shape}")

        for name, arr in [
            ('X_train', X_train), ('y_train', y_train),
            ('X_val',   X_val),   ('y_val',   y_val),
            ('X_test',  X_test),  ('y_test',  y_test),
        ]:
            if np.isnan(arr).any() or np.isinf(arr).any():
                raise ValueError(f"{name} contains NaN or inf after scaling")

        print("✓ Data validation passed — all values are valid float32")

        # ── 6. Persist scaler ─────────────────────────────────────────
        if save_scaler:
            os.makedirs(self.config['paths']['model_dir'], exist_ok=True)
            scaler_path = os.path.join(
                self.config['paths']['model_dir'], f'scaler_{dataset_name}.pkl'
            )
            joblib.dump(target_scaler, scaler_path)

        return {
            'X_train':     X_train,
            'y_train':     y_train,
            'X_val':       X_val,
            'y_val':       y_val,
            'X_test':      X_test,
            'y_test':      y_test,
            'feature_cols': feature_cols,
            'target_cols':  target_cols,
            'feature_dim':  n_features,
            'target_dim':   n_targets,
            'input_dim':    n_features + n_targets,
            'scaler':       target_scaler,
        }

    def prepare_all_datasets(self) -> Dict[str, Dict]:
        datasets = {}
        for dataset_config in self.config['data']['datasets']:
            dataset_name = dataset_config['name']
            try:
                print(f"\n{'='*60}")
                print(f"Preparing: {dataset_name}")
                print(f"{'='*60}")
                datasets[dataset_name] = self.prepare_dataset(dataset_config)
            except Exception as e:
                print(f"✗ Error preparing {dataset_name}: {e}")
                import traceback
                traceback.print_exc()
        return datasets

Overwriting data/preprocessing.py


In [ ]:
#@title train.py
%%writefile experiments/train.py
#!/usr/bin/env python3
"""
Optimized Trainer — T4 GPU, AMP, combined forward, AdamW + cosine schedule.
"""

import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader
import numpy as np
import os
import json
import gc
import math
from tqdm import tqdm
from typing import Dict, Optional

from data.dataset import TimeSeriesDataset
from models.llm_model import EnhancedTimeSeriesLLM
from models.losses import CombinedLoss
from utils.metrics import evaluate_model, print_metrics
from utils.helpers import save_checkpoint, load_checkpoint


class EarlyStopping:
    def __init__(self, patience: int = 12, min_delta: float = 0.0001):
        self.patience    = patience
        self.min_delta   = min_delta
        self.counter     = 0
        self.best_loss   = None
        self.should_stop = False

    def __call__(self, val_loss: float) -> bool:
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        else:
            self.best_loss = val_loss
            self.counter   = 0
        return self.should_stop


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)


def _get_projection_head(model: nn.Module):
    """
    Resolve projection_head through several wrapper layers:
      1. Direct attribute (EnhancedTimeSeriesLLM, compiled model)
      2. torch.compile _orig_mod wrapper
      3. AblationLLM.base_model (AblationLLM wraps EnhancedTimeSeriesLLM)
      4. AblationLLM._orig_mod.base_model (compiled AblationLLM)
    """
    # Direct
    if hasattr(model, 'projection_head'):
        return model.projection_head
    # torch.compile wrapper
    if hasattr(model, '_orig_mod'):
        inner = model._orig_mod
        if hasattr(inner, 'projection_head'):
            return inner.projection_head
        # AblationLLM inside compile wrapper
        if hasattr(inner, 'base_model') and hasattr(inner.base_model, 'projection_head'):
            return inner.base_model.projection_head
    # AblationLLM (not compiled)
    if hasattr(model, 'base_model') and hasattr(model.base_model, 'projection_head'):
        return model.base_model.projection_head
    raise AttributeError(
        f"Cannot find projection_head on {type(model).__name__} "
        "or its .base_model / ._orig_mod wrappers"
    )


class Trainer:
    def __init__(self, model: nn.Module, config: Dict, device: torch.device,
                 accum_steps: int = 2):
        torch.backends.cudnn.benchmark        = True
        torch.backends.cudnn.deterministic    = False
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32       = True

        self.model       = model.to(device)
        self.config      = config
        self.device      = device
        self.accum_steps = accum_steps

        self.use_amp = (device.type == 'cuda')
        self.scaler  = GradScaler('cuda', enabled=self.use_amp)

        loss_cfg = config['loss']
        scta_cfg = {
            'lambda_softcl':     loss_cfg['lambda_softcl'],
            'lambda_crossmodal': loss_cfg['lambda_crossmodal'],
            'temperature':       loss_cfg.get('temperature', 0.07),
        }
        self.criterion = CombinedLoss(
            lambda_forecast = loss_cfg.get('lambda_forecast', 1.0),
            lambda_scta     = loss_cfg.get('lambda_scta', 0.3),
            lambda_spectral = loss_cfg.get('lambda_spectral', 0.05),
            scta_config     = scta_cfg,
        )

        train_cfg = config['training']
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr           = train_cfg['learning_rate'],
            weight_decay = 1e-4,
            betas        = (0.9, 0.95),
        )
        self.scheduler = None

        self.early_stopping = EarlyStopping(
            patience  = train_cfg.get('patience', 12),
            min_delta = train_cfg.get('min_delta', 0.0001),
        )

        self.train_losses  = []
        self.val_losses    = []
        self.best_val_loss          = float('inf')
        self.best_forecast_val_loss  = float('inf')

    def _to(self, batch):
        return [t.to(self.device, non_blocking=True) if torch.is_tensor(t) else t
                for t in batch]

    def _optimizer_step(self):
        """Unscale → clip → step → update → zero_grad."""
        self.scaler.unscale_(self.optimizer)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad(set_to_none=True)
        if self.scheduler is not None:
            self.scheduler.step()

    def train_epoch(self, train_loader: DataLoader) -> Dict[str, float]:
        self.model.train()
        totals = dict(total=0., forecast=0., scta=0., softcl=0.,
                      crossmodal=0., spectral=0.)

        self.optimizer.zero_grad(set_to_none=True)

        # BUG 10 FIX: resolve projection_head once, safely handles compiled models
        proj_head = _get_projection_head(self.model)

        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc='Train', leave=False)

        for step, batch in pbar:
            X_i, X_j, T_emb, temporal_weight, y = self._to(batch)

            with autocast('cuda', enabled=self.use_amp, dtype=torch.float16):
                y_pred, emb_i = self.model(X_i, return_embedding=True)
                _,      emb_j = self.model(X_j, return_embedding=True)
                proj_i = proj_head(emb_i)
                proj_j = proj_head(emb_j)
                loss, ld = self.criterion(
                    y_pred, y, proj_i, proj_j, T_emb, temporal_weight
                )
                loss = loss / self.accum_steps

            self.scaler.scale(loss).backward()

            if (step + 1) % self.accum_steps == 0:
                self._optimizer_step()

            for k in totals:
                totals[k] += ld.get(k, 0.)

            pbar.set_postfix(loss=f"{ld['total']:.4f}",
                             fore=f"{ld['forecast']:.4f}")

        # BUG 9 FIX: flush any remaining accumulated gradients from the last
        # partial accumulation window (triggered when len(loader) % accum_steps != 0).
        remainder = len(train_loader) % self.accum_steps
        if remainder != 0:
            self._optimizer_step()

        n = len(train_loader)
        return {k: v / n for k, v in totals.items()}

    def validate(self, val_loader: DataLoader) -> Dict[str, float]:
        self.model.eval()
        totals = dict(total=0., forecast=0., scta=0., softcl=0.,
                      crossmodal=0., spectral=0.)

        proj_head = _get_projection_head(self.model)

        with torch.no_grad():
            for batch in val_loader:
                X_i, X_j, T_emb, temporal_weight, y = self._to(batch)
                with autocast('cuda', enabled=self.use_amp, dtype=torch.float16):
                    y_pred, emb_i = self.model(X_i, return_embedding=True)
                    _,      emb_j = self.model(X_j, return_embedding=True)
                    proj_i = proj_head(emb_i)
                    proj_j = proj_head(emb_j)
                    _, ld = self.criterion(
                        y_pred, y, proj_i, proj_j, T_emb, temporal_weight
                    )
                for k in totals:
                    totals[k] += ld.get(k, 0.)

        n = len(val_loader)
        return {k: v / n for k, v in totals.items()}

    def train(self, train_loader: DataLoader, val_loader: DataLoader,
              dataset_name: str) -> Dict:
        train_cfg = self.config['training']
        epochs    = train_cfg['epochs']
        model_dir = self.config['paths']['model_dir']
        os.makedirs(model_dir, exist_ok=True)

        # total_steps = (len(train_loader) // self.accum_steps) * epochs
        total_steps = math.ceil(len(train_loader) / self.accum_steps) * epochs
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr           = train_cfg['learning_rate'],
            total_steps      = max(total_steps, 1),
            pct_start        = 0.1,
            anneal_strategy  = 'cos',
            div_factor       = 25.0,
            final_div_factor = 1e4,
        )

        history = {'train': [], 'val': []}

        for epoch in range(1, epochs + 1):
            train_losses = self.train_epoch(train_loader)
            val_losses   = self.validate(val_loader)

            self.train_losses.append(train_losses['total'])
            self.val_losses.append(val_losses['total'])
            history['train'].append(train_losses)
            history['val'].append(val_losses)

            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"train={train_losses['total']:.4f} "
                  f"(fore={train_losses['forecast']:.4f} "
                  f"scta={train_losses['scta']:.4f} "
                  f"spec={train_losses.get('spectral', 0):.4f}) | "
                  f"val={val_losses['total']:.4f}")

            # Checkpoint on val FORECAST loss — not total (total is dominated
            # by SCTA/spectral auxiliary terms that don't reflect forecast quality)
            val_fore = val_losses['forecast']
            if val_fore < self.best_forecast_val_loss:
                self.best_forecast_val_loss = val_fore
                ckpt_path = os.path.join(model_dir, f'best_model_{dataset_name}.pth')
                save_model = (self.model._orig_mod
                              if hasattr(self.model, '_orig_mod') else self.model)
                save_checkpoint(save_model, self.optimizer, epoch,
                                val_fore, ckpt_path)
                print(f"  ✓ Saved best → {ckpt_path}  "
                      f"(val_fore={val_fore:.4f} | val_total={val_losses['total']:.4f})")

            if self.early_stopping(val_losses['forecast']):
                print(f"Early stopping at epoch {epoch}")
                break

        return history

    def load_checkpoint(self, dataset_name: str):
        path = os.path.join(self.config['paths']['model_dir'],
                            f'best_model_{dataset_name}.pth')
        load_model = (self.model._orig_mod
                      if hasattr(self.model, '_orig_mod') else self.model)
        load_checkpoint(path, load_model, device=self.device)


# ---------------------------------------------------------------------------

def train_on_dataset(dataset_name: str, data_dict: Dict, config: Dict,
                     device: torch.device) -> Dict:
    batch_size = config['training']['batch_size']
    d_model    = config['model']['llm']['d_model']

    train_dataset = TimeSeriesDataset(
        data_dict['X_train'], data_dict['y_train'],
        d_model, augmentation=True, dataset_name=dataset_name
    )
    val_dataset = TimeSeriesDataset(
        data_dict['X_val'], data_dict['y_val'],
        d_model, augmentation=False, dataset_name=dataset_name
    )
    test_dataset = TimeSeriesDataset(
        data_dict['X_test'], data_dict['y_test'],
        d_model, augmentation=False, dataset_name=dataset_name
    )

    loader_kw = dict(
        pin_memory         = True,
        num_workers        = 2,
        prefetch_factor    = 4,
        persistent_workers = True,
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True,  **loader_kw)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                              shuffle=False, **loader_kw)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                              shuffle=False, **loader_kw)

    model = EnhancedTimeSeriesLLM(
        input_dim         = data_dict['input_dim'],
        output_dim        = data_dict['target_dim'],
        prediction_length = config['data'].get('prediction_length', 96),
        **config['model']['llm']
    )

    try:
        model = torch.compile(model, mode='default')
        print("✓ torch.compile enabled")
    except Exception as e:
        print(f"torch.compile skipped: {e}")

    accum_steps = config['training'].get('accum_steps', 2)
    trainer     = Trainer(model, config, device, accum_steps=accum_steps)
    history     = trainer.train(train_loader, val_loader, dataset_name)

    trainer.load_checkpoint(dataset_name)

    # BUG 12 FIX: pass return_embedding=False explicitly so compiled model
    # receives the keyword arg unambiguously.
    test_metrics, y_pred, y_true = evaluate_model(
        trainer.model, test_loader, device, data_dict.get('scaler')
    )
    print_metrics(test_metrics, prefix=f"{dataset_name} TEST — ")

    return {
        'test_metrics':     test_metrics,
        'training_history': history,
        'model':            trainer.model,
    }


Overwriting experiments/train.py


In [ ]:
#@title encoder.py
%%writefile models/encoder.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple


class TimeSeriesEncoder(nn.Module):
    """
    Time-Series Encoder for extracting representations from time series data.
    Uses convolutional layers for local pattern extraction.
    """

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int,
                 num_layers: int = 3, kernel_size: int = 3, dropout: float = 0.1):
        """
        Args:
            input_dim: Number of input features
            hidden_dim: Hidden dimension for conv layers
            output_dim: Output embedding dimension
            num_layers: Number of convolutional layers
            kernel_size: Kernel size for convolutions
            dropout: Dropout rate
        """
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        # Convolutional layers
        self.conv_layers = nn.ModuleList()

        # First layer
        self.conv_layers.append(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=kernel_size,
                     padding=kernel_size//2)
        )

        # Middle layers
        for _ in range(num_layers - 2):
            self.conv_layers.append(
                nn.Conv1d(hidden_dim, hidden_dim, kernel_size=kernel_size,
                         padding=kernel_size//2)
            )

        # Last layer
        self.conv_layers.append(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=kernel_size,
                     padding=kernel_size//2)
        )

        # Batch normalization layers
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm1d(hidden_dim) for _ in range(num_layers)
        ])

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Pooling
        self.pool = nn.AdaptiveAvgPool1d(1)

        # Final projection
        self.fc = nn.Linear(hidden_dim, output_dim)

        # Activation
        self.activation = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x: Input tensor [batch_size, seq_len, input_dim]

        Returns:
            Embedding tensor [batch_size, output_dim]
        """
        # Transpose for Conv1d: [batch, features, seq_len]
        x = x.transpose(1, 2)

        # Apply convolutional layers
        for conv, bn in zip(self.conv_layers, self.batch_norms):
            x = conv(x)
            x = bn(x)
            x = self.activation(x)
            x = self.dropout(x)

        # Global pooling
        x = self.pool(x).squeeze(-1)  # [batch, hidden_dim]

        # Final projection
        x = self.fc(x)  # [batch, output_dim]

        return x


class TemporalConvNet(nn.Module):
    """
    Temporal Convolutional Network (TCN) for time series encoding.
    Uses dilated causal convolutions for long-range dependencies.
    """

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int,
                 num_layers: int = 4, kernel_size: int = 3, dropout: float = 0.1):
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        layers = []
        num_levels = num_layers

        for i in range(num_levels):
            dilation = 2 ** i
            in_channels = input_dim if i == 0 else hidden_dim
            out_channels = hidden_dim

            layers.append(
                TemporalBlock(
                    in_channels, out_channels, kernel_size,
                    stride=1, dilation=dilation, padding=(kernel_size-1) * dilation,
                    dropout=dropout
                )
            )

        self.network = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input [batch, seq_len, input_dim]
        Returns:
            Output [batch, output_dim]
        """
        # Transpose: [batch, input_dim, seq_len]
        x = x.transpose(1, 2)

        # Apply TCN
        x = self.network(x)

        # Pool and project
        x = self.pool(x).squeeze(-1)
        x = self.fc(x)

        return x


class TemporalBlock(nn.Module):
    """Single block in TCN with residual connection."""

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int,
                 stride: int, dilation: int, padding: int, dropout: float = 0.1):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                              stride=stride, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        # Residual connection
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x

        # First conv block
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        # Second conv block
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        # Adjust output length to match input (causal convolution padding)
        if out.size(2) != x.size(2):
            out = out[:, :, :x.size(2)]

        # Residual connection
        if self.downsample is not None:
            residual = self.downsample(residual)

        return self.relu(out + residual)


class LLMTextEncoder(nn.Module):
    """
    Text encoder using transformer architecture.
    This is a simplified version - in practice, you'd use a pre-trained LLM.
    """

    def __init__(self, vocab_size: int, hidden_dim: int, output_dim: int,
                 n_heads: int = 4, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        # Token embedding
        self.embedding = nn.Embedding(vocab_size, hidden_dim)

        # Positional encoding
        self.pos_encoding = PositionalEncoding(hidden_dim, dropout)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)

        # Output projection
        self.fc_embedding = nn.Linear(hidden_dim, output_dim)  # For contrastive learning
        self.fc_logits = nn.Linear(hidden_dim, vocab_size)     # For language modeling

    def forward(self, x: torch.Tensor, return_logits: bool = False) -> torch.Tensor:
        """
        Args:
            x: Input token ids [batch, seq_len]
            return_logits: If True, return logits for LM; else return embeddings

        Returns:
            Either embeddings [batch, output_dim] or logits [batch, vocab_size]
        """
        # Embed tokens
        x = self.embedding(x) * math.sqrt(self.hidden_dim)

        # Add positional encoding
        x = self.pos_encoding(x)

        # Transform
        x = self.transformer(x)

        # Pool: use CLS token (first token) or mean pooling
        x = x.mean(dim=1)  # Mean pooling

        if return_logits:
            return self.fc_logits(x)
        else:
            return self.fc_embedding(x)


class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding for non-recurrent transformers.
    (Retained from original notebook for completeness, though often replaced by learned embeddings in modern TS models)
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)



class MultiModalEncoder(nn.Module):
    """
    Combined encoder for time series and text modalities.
    Projects both to a shared embedding space.
    """

    def __init__(self, ts_input_dim: int, text_vocab_size: int,
                 ts_hidden_dim: int, text_hidden_dim: int,
                 shared_dim: int, dropout: float = 0.1):
        super().__init__()

        # Time series encoder
        self.ts_encoder = TimeSeriesEncoder(
            ts_input_dim, ts_hidden_dim, shared_dim, dropout=dropout
        )

        # Text encoder (simplified - use pre-trained in production)
        self.text_encoder = LLMTextEncoder(
            text_vocab_size, text_hidden_dim, shared_dim, dropout=dropout
        )

        # Projection heads for contrastive learning
        self.ts_projection = nn.Sequential(
            nn.Linear(shared_dim, shared_dim),
            nn.ReLU(),
            nn.Linear(shared_dim, shared_dim)
        )

        self.text_projection = nn.Sequential(
            nn.Linear(shared_dim, shared_dim),
            nn.ReLU(),
            nn.Linear(shared_dim, shared_dim)
        )

    def forward_ts(self, ts: torch.Tensor, project: bool = False) -> torch.Tensor:
        """Encode time series."""
        emb = self.ts_encoder(ts)
        if project:
            emb = self.ts_projection(emb)
        return emb

    def forward_text(self, text: torch.Tensor, project: bool = False) -> torch.Tensor:
        """Encode text."""
        emb = self.text_encoder(text, return_logits=False)
        if project:
            emb = self.text_projection(emb)
        return emb

    def forward(self, ts: torch.Tensor, text: torch.Tensor,
                project: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode both modalities."""
        ts_emb = self.forward_ts(ts, project)
        text_emb = self.forward_text(text, project)
        return ts_emb, text_emb

Overwriting models/encoder.py


In [ ]:
#@title hsqp.py
%%writefile models/hsqp.py
import torch
import torch.nn as nn
import numpy as np
from sklearn.cluster import KMeans
from typing import List, Tuple, Dict, Union, Optional
import math

class TimeSeriesPatching:
    """
    Class for creating patches from time series data (Step 2 in HSQP).
    """
    def __init__(self, patch_length: int = 24, stride: int = 12, overlap: bool = True):
        """
        Initialize the patching parameters.

        Args:
            patch_length: Length of each patch
            stride: Step size between patches (if overlap=True)
            overlap: Whether patches should overlap
        """
        self.patch_length = patch_length
        self.stride = stride if overlap else patch_length
        self.overlap = overlap

    def create_patches(self, time_series: np.ndarray) -> np.ndarray:
        """
        Create patches from a time series.

        Args:
            time_series: Time series data of shape [batch_size, seq_length, features]
                         or [seq_length, features] or [seq_length]

        Returns:
            Patches of shape [batch_size, num_patches, patch_length, features]
                         or [num_patches, patch_length, features]
                         or [num_patches, patch_length]
        """
        # Handle different input shapes
        original_shape = time_series.shape
        if len(original_shape) == 1:
            # Convert [seq_length] to [seq_length, 1]
            time_series = time_series.reshape(-1, 1)
            seq_length, features = time_series.shape
            batch_size = None
        elif len(original_shape) == 2:
            # [seq_length, features]
            seq_length, features = time_series.shape
            batch_size = None
        else:
            # [batch_size, seq_length, features]
            batch_size, seq_length, features = time_series.shape

        # Calculate number of patches
        num_patches = (seq_length - self.patch_length) // self.stride + 1

        if batch_size is None:
            # Initialize patches array
            patches = np.zeros((num_patches, self.patch_length, features))

            # Create patches
            for i in range(num_patches):
                start_idx = i * self.stride
                end_idx = start_idx + self.patch_length
                patches[i] = time_series[start_idx:end_idx]

            # Restore original dimensionality if input was 1D
            if len(original_shape) == 1:
                patches = patches.reshape(num_patches, self.patch_length)
        else:
            # Initialize patches array for batched data
            patches = np.zeros((batch_size, num_patches, self.patch_length, features))

            # Create patches
            for b in range(batch_size):
                for i in range(num_patches):
                    start_idx = i * self.stride
                    end_idx = start_idx + self.patch_length
                    patches[b, i] = time_series[b, start_idx:end_idx]

        return patches

    def merge_patches(self, patches: np.ndarray, original_length: Optional[int] = None) -> np.ndarray:
        """
        Merge patches back into a time series.
        For overlapping regions, values are averaged.

        Args:
            patches: Patches of shape [batch_size, num_patches, patch_length, features]
                     or [num_patches, patch_length, features]
                     or [num_patches, patch_length]
            original_length: Original sequence length (optional)

        Returns:
            Reconstructed time series
        """
        # Handle different input shapes
        original_shape = patches.shape
        if len(original_shape) == 2:
            # [num_patches, patch_length] -> [num_patches, patch_length, 1]
            patches = patches.reshape(original_shape[0], original_shape[1], 1)
            num_patches, patch_length, features = patches.shape
            batch_size = None
        elif len(original_shape) == 3:
            # [num_patches, patch_length, features]
            num_patches, patch_length, features = patches.shape
            batch_size = None
        else:
            # [batch_size, num_patches, patch_length, features]
            batch_size, num_patches, patch_length, features = patches.shape

        # Calculate reconstructed sequence length
        if original_length is None:
            seq_length = (num_patches - 1) * self.stride + patch_length
        else:
            seq_length = original_length

        if batch_size is None:
            # Initialize reconstructed time series and count array for averaging
            reconstructed = np.zeros((seq_length, features))
            counts = np.zeros((seq_length, features))

            # Merge patches
            for i in range(num_patches):
                start_idx = i * self.stride
                end_idx = start_idx + patch_length
                reconstructed[start_idx:end_idx] += patches[i]
                counts[start_idx:end_idx] += 1

            # Average overlapping regions
            reconstructed = reconstructed / np.maximum(counts, 1)

            # Restore original dimensionality if input was 2D
            if len(original_shape) == 2:
                reconstructed = reconstructed.reshape(seq_length)
        else:
            # Initialize reconstructed time series and count array for batched data
            reconstructed = np.zeros((batch_size, seq_length, features))
            counts = np.zeros((batch_size, seq_length, features))

            # Merge patches
            for b in range(batch_size):
                for i in range(num_patches):
                    start_idx = i * self.stride
                    end_idx = start_idx + patch_length
                    reconstructed[b, start_idx:end_idx] += patches[b, i]
                    counts[b, start_idx:end_idx] += 1

            # Average overlapping regions
            reconstructed = reconstructed / np.maximum(counts, 1)

        return reconstructed


class ABBASymbolicAggregation:
    """
    Implementation of ABBA (Aggregation-Based Amplitude Scaling) for symbolic pattern extraction (Step 3 in HSQP).
    This is a simplified implementation based on the fABBA library concepts.
    """
    def __init__(self, tol: float = 0.1, alpha: float = 0.1, sorting: str = '2-norm', scl: float = 1, k: int = 10):
        """
        Initialize ABBA parameters.

        Args:
            tol: Tolerance for ABBA compression
            alpha: Parameter for ABBA digitization
            sorting: Sorting method for clustering ('2-norm' or 'amplitude')
            scl: Scaling factor
            k: Number of symbols/clusters for ABBA
        """
        self.tol = tol
        self.alpha = alpha
        self.sorting = sorting
        self.scl = scl
        self.k = k
        self.parameters = None  # Stores cluster centers after fitting

    def compress(self, ts: np.ndarray) -> List[Tuple[float, float]]:
        """
        Compresses a time series into a sequence of polygonal segments (pieces).
        Each piece is (length, increment).
        """
        pieces = []
        i = 0
        while i < len(ts) - 1:
            start_val = ts[i]
            j = i + 1
            while j < len(ts) and abs(ts[j] - start_val) < self.tol:
                j += 1

            if j == i + 1:
                # Single point segment
                pieces.append((1.0, ts[j] - ts[i] if j < len(ts) else 0.0))
                i = j
            else:
                # Segment of length > 1
                length = j - i
                increment = ts[j - 1] - ts[i]
                pieces.append((float(length), increment))
                i = j - 1 # Overlap by 1 for next segment start

        return pieces

    def digitize(self, pieces: List[Tuple[float, float]]) -> Tuple[List[str], np.ndarray]:
        """
        Digitizes the polygonal segments using k-means clustering.
        """
        if not pieces:
            return [], np.array([])

        # Convert pieces to numpy array for clustering
        piece_array = np.array(pieces)

        # Apply scaling if necessary
        if self.scl != 1:
            piece_array = piece_array / self.scl

        # K-means clustering
        kmeans = KMeans(n_clusters=self.k, random_state=0, n_init=10)
        kmeans.fit(piece_array)
        centers = kmeans.cluster_centers_

        # Store parameters
        self.parameters = {
            'centers': centers,
            'tol': self.tol,
            'alpha': self.alpha,
            'scl': self.scl,
            'k': self.k
        }

        # Assign symbols (a, b, c, ...)
        labels = kmeans.predict(piece_array)
        symbols = [chr(97 + label) for label in labels]

        return symbols, piece_array


    def fit_transform(self, ts: np.ndarray) -> str:
        """
        Apply ABBA transformation to time series.

        Args:
            ts: Time series data

        Returns:
            Symbolic representation of the time series
        """
        pieces = self.compress(ts)
        symbols, _ = self.digitize(pieces)
        return ''.join(symbols)

    def inverse_transform(self, string: str, initial_value: float) -> np.ndarray:
        """
        Convert symbolic representation back to time series.

        Args:
            string: Symbolic representation
            initial_value: Initial value of the time series

        Returns:
            Reconstructed time series
        """
        if self.parameters is None:
            raise ValueError("ABBA model must be fitted before inverse transform")

        # Convert string to cluster indices
        indices = [ord(s) - 97 for s in string]

        # Get cluster centers
        centers = self.parameters['centers']

        # Scale back if needed
        if self.scl != 1:
            centers = centers * self.scl

        # Reconstruct pieces
        pieces = [tuple(centers[idx]) for idx in indices]

        # Reconstruct time series
        ts_recon = [initial_value]
        for length, increment in pieces:
            # Convert float length to integer
            length = int(round(length))
            if length < 1:
                length = 1

            # Create linear segment
            if length == 1:
                ts_recon.append(ts_recon[-1] + increment)
            else:
                # Linear interpolation for the segment
                start_val = ts_recon[-1]
                end_val = start_val + increment
                segment = np.linspace(start_val, end_val, length + 1)[1:] # Exclude start_val, include end_val
                ts_recon.extend(segment)

        return np.array(ts_recon)


class FeatureQuantization:
    """
    Quantization of ABBA-derived features for efficiency optimization (Step 4 in HSQP).
    """
    def __init__(self, bit_width: int = 8, method: str = 'affine', block_size: int = 32):
        """
        Initialize quantization parameters.

        Args:
            bit_width: Target bit width (e.g., 8 for INT8, 4 for INT4)
            method: Quantization method ('affine', 'abs_max')
            block_size: Block size for block-wise quantization (not fully implemented here, but kept for signature)
        """
        self.bit_width = bit_width
        self.method = method
        self.block_size = block_size
        self.scale = None
        self.zero_point = None

        # Calculate quantization range
        self.qmin = -(2 ** (bit_width - 1))
        self.qmax = 2 ** (bit_width - 1) - 1

    def quantize(self, features: np.ndarray) -> np.ndarray:
        """
        Quantize features to lower precision.
        """
        if self.method == 'abs_max':
            # Absolute max quantization
            abs_max = np.max(np.abs(features))
            if abs_max == 0:
                abs_max = 1.0  # Avoid division by zero

            self.scale = self.qmax / abs_max
            self.zero_point = 0

            # Quantize
            q_features = np.round(features * self.scale)
            q_features = np.clip(q_features, self.qmin, self.qmax)

        elif self.method == 'affine':
            # Affine quantization
            f_min = np.min(features)
            f_max = np.max(features)

            if f_min == f_max:
                self.scale = 1.0
                self.zero_point = 0
            else:
                self.scale = (self.qmax - self.qmin) / (f_max - f_min)
                self.zero_point = self.qmin - round(f_min * self.scale)

            # Quantize
            q_features = np.round(features * self.scale + self.zero_point)
            q_features = np.clip(q_features, self.qmin, self.qmax)

        else:
            raise ValueError(f"Unknown quantization method: {self.method}")

        return q_features.astype(np.int8 if self.bit_width <= 8 else np.int16)

    def dequantize(self, q_features: np.ndarray) -> np.ndarray:
        """
        Dequantize features back to original precision.
        """
        if self.scale is None or (self.method == 'affine' and self.zero_point is None):
            raise ValueError("Quantization parameters not set. Call quantize() first.")

        if self.method == 'abs_max':
            return q_features / self.scale
        elif self.method == 'affine':
            return (q_features - self.zero_point) / self.scale
        else:
            raise ValueError(f"Unknown quantization method: {self.method}")


class HSQP:
    """
    Hierarchical Symbolic-Quantized Patching (HSQP) for time-series tokenization.
    This class acts as the integrated tokenizer and embedding layer.
    """
    def __init__(self,
                 patch_length: int = 24,
                 stride: int = 12,
                 tol: float = 0.01,
                 alpha: float = 0.01,
                 k: int = 26,
                 bit_width: int = 8,
                 quant_method: str = 'affine',
                 embedding_dim: int = 64):
        """
        Initialize HSQP parameters.

        Args:
            patch_length: Length of each patch
            stride: Step size between patches
            tol: Tolerance for ABBA compression
            alpha: Parameter for ABBA digitization
            k: Number of symbols/clusters for ABBA
            bit_width: Target bit width for quantization
            quant_method: Quantization method
            embedding_dim: Dimension for LLM embedding
        """
        self.patching = TimeSeriesPatching(patch_length=patch_length, stride=stride)
        self.abba = ABBASymbolicAggregation(tol=tol, alpha=alpha, k=k)
        self.quantization = FeatureQuantization(bit_width=bit_width, method=quant_method)
        self.embedding_dim = embedding_dim


        self.embedding = nn.Linear(2, embedding_dim)

    def fit_transform(self, time_series: np.ndarray) -> Tuple[List[str], np.ndarray, List[List[Tuple[float, float]]], torch.Tensor]:
        """
        Apply HSQP transformation to time series.

        Args:
            time_series: Input time series data

        Returns:
            symbols_list: List of symbolic representations for each patch
            quantized_features: Quantized ABBA-derived features (concatenated)
            pieces_list: List of polygonal segments for each patch
            embeddings: PyTorch tensor of embeddings [num_patches, embedding_dim]
        """
        # Step 2: Initial Patching
        patches = self.patching.create_patches(time_series)

        # Handle different input shapes
        if len(patches.shape) == 4:  # [batch_size, num_patches, patch_length, features]
            raise NotImplementedError("Batched HSQP transformation is complex and not fully implemented in this simplified version.")
        else:  # [num_patches, patch_length, features] or [num_patches, patch_length]
            num_patches = patches.shape[0]
            # If input was [num_patches, patch_length], reshape to [num_patches, patch_length, 1]
            if len(patches.shape) == 2:
                patches = patches.reshape(num_patches, patches.shape[1], 1)
            features_dim = patches.shape[2]

        # Step 3: ABBA Symbolic Aggregation and Feature Extraction
        symbols_list = []
        pieces_list = []
        all_pieces_array = []

        for i in range(num_patches):

            ts_patch = patches[i, :, 0]

            # Compress and Digitize
            pieces = self.abba.compress(ts_patch)
            symbols, piece_array = self.abba.digitize(pieces) # piece_array is [N_pieces, 2] (length, increment)

            symbols_list.append(''.join(symbols))
            pieces_list.append(pieces)
            all_pieces_array.append(piece_array)

        # Concatenate all pieces for quantization and embedding
        all_pieces_array = np.concatenate(all_pieces_array, axis=0) # [Total_N_pieces, 2]

        # Step 4: Quantization of ABBA-Derived Features
        quantized_features = self.quantization.quantize(all_pieces_array)

        # Step 5: LLM Tokenization and Embedding

        # Convert features to PyTorch tensor
        piece_tensor = torch.FloatTensor(all_pieces_array)


        embeddings = self.embedding(piece_tensor)

        return symbols_list, quantized_features, pieces_list, embeddings

    def inverse_transform(self, symbols_list: List[str], initial_value: float, original_length: int) -> np.ndarray:
        """
        Inverse HSQP transformation to reconstruct the time series.
        This is a simplified version that only reconstructs the first feature dimension.
        """
        reconstructed_ts = []
        current_value = initial_value

        for symbol_string in symbols_list:
            # Reconstruct the patch from the symbolic string
            patch_recon = self.abba.inverse_transform(symbol_string, current_value)


            if len(patch_recon) > 1:
                reconstructed_ts.extend(patch_recon[1:])
                current_value = patch_recon[-1]


        final_ts = np.array(reconstructed_ts)

        if len(final_ts) > original_length:
            final_ts = final_ts[:original_length]
        elif len(final_ts) < original_length:
            # Simple padding with the last value
            padding = np.full(original_length - len(final_ts), final_ts[-1] if len(final_ts) > 0 else initial_value)
            final_ts = np.concatenate([final_ts, padding])

        return final_ts

Overwriting models/hsqp.py


In [ ]:
#@title llm_model.py
%%writefile models/llm_model.py
import torch
import torch.nn as nn
import math
from typing import Optional


# ---------------------------------------------------------------------------
# RevIN — Reversible Instance Normalization
# ---------------------------------------------------------------------------

class RevIN(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if affine:
            self.gamma = nn.Parameter(torch.ones(1, 1, num_features))
            self.beta  = nn.Parameter(torch.zeros(1, 1, num_features))
        self._mean: Optional[torch.Tensor] = None
        self._std:  Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor, mode: str) -> torch.Tensor:
        if mode == 'norm':
            return self._normalize(x)
        elif mode == 'denorm':
            return self._denormalize(x)
        else:
            raise ValueError(f"RevIN mode must be 'norm' or 'denorm', got '{mode}'")

    def _normalize(self, x: torch.Tensor) -> torch.Tensor:
        self._mean = x.mean(dim=1, keepdim=True).detach()   # [B, 1, C]
        self._std  = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
        x_norm = (x - self._mean) / self._std
        if self.affine:
            x_norm = x_norm * self.gamma + self.beta
        return x_norm

    def _denormalize(self, x: torch.Tensor) -> torch.Tensor:
        if self._mean is None or self._std is None:
            raise RuntimeError("RevIN.denorm called before norm — call norm first.")
        if self.affine:
            x = (x - self.beta) / (self.gamma + self.eps)
        return x * self._std + self._mean


# ---------------------------------------------------------------------------
# SimCLR-style Projection Head
# ---------------------------------------------------------------------------

class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: Optional[int] = None, out_dim: int = 128):
        super().__init__()
        hidden_dim = hidden_dim or in_dim * 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.net(x)
        return nn.functional.normalize(z, dim=-1)


# ---------------------------------------------------------------------------
# Positional Encoding
# BUG 3 + 4 FIXED:
#   - pe stored as [1, max_len, d_model] (batch_first compatible)
#   - forward indexes dim=1 (seq dim) not dim=0 (batch dim)
# ---------------------------------------------------------------------------

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding — batch_first compatible."""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # BUG 4 FIX: store as [1, max_len, d_model] so it broadcasts correctly
        # against x of shape [B, T, d_model] (batch_first=True layout).
        # The original code did unsqueeze(0).transpose(0,1) → [max_len, 1, d_model]
        # which mismatches batch_first tensors.
        pe = pe.unsqueeze(0)          # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # BUG 3 FIX: x is [B, T, d_model] with batch_first=True.
        # x.size(0) = B (batch), x.size(1) = T (sequence length).
        # We must slice pe along the sequence dimension (dim=1), not dim=0.
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ---------------------------------------------------------------------------
# Main model
# BUG 2 FIXED: added use_hsqp, patch_len, patch_stride to __init__ signature.
# They were referenced as variables inside __init__ but were never declared
# as parameters, causing immediate NameError on construction.
# ---------------------------------------------------------------------------

class EnhancedTimeSeriesLLM(nn.Module):
    """
    Transformer-based time series forecaster with:
      - RevIN for instance-level normalisation / denormalisation
      - SimCLR ProjectionHead for contrastive learning in projection space
      - Optional HSQP patch tokenisation (use_hsqp=True)
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        prediction_length: int = 96,
        d_model: int = 512,
        n_heads: int = 8,
        n_layers: int = 8,
        d_ff: int = 2048,
        dropout: float = 0.1,
        proj_out_dim: int = 128,
        revin_affine: bool = True,
        # BUG 2 FIX: these three were used inside __init__ without being parameters
        use_hsqp: bool = True,
        patch_len: int = 24,
        patch_stride: int = 12,
    ):
        super().__init__()
        self.input_dim         = input_dim
        self.output_dim        = output_dim
        self.d_model           = d_model
        self.prediction_length = prediction_length
        self.use_hsqp          = use_hsqp
        self.patch_len         = patch_len
        self.patch_stride      = patch_stride

        # RevIN on target channels only
        self.revin = RevIN(num_features=output_dim, affine=revin_affine)

        # Input projection — either patch-based or per-timestep
        if use_hsqp:
            self.patch_proj = nn.Linear(patch_len, d_model)
        else:
            self.input_projection = nn.Linear(input_dim, d_model)

        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation='gelu', batch_first=True,
            norm_first=True,   # Pre-LN: more stable at depth
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, n_layers)

        self.forecasting_head = nn.Linear(d_model, output_dim * prediction_length)

        self.projection_head = ProjectionHead(
            in_dim=d_model,
            hidden_dim=d_model * 2,
            out_dim=proj_out_dim,
        )

    # ------------------------------------------------------------------

    def _encode(self, src: torch.Tensor, src_mask=None) -> torch.Tensor:
        if self.use_hsqp:
            # Patchify: [B, T, C] → [B, P, d_model]
            B, T, C = src.shape
            x_t     = src.permute(0, 2, 1)                              # [B, C, T]
            # patches = x_t.unfold(-1, self.patch_len, self.patch_stride) # [B, C, P, L]
            # tokens  = patches.mean(dim=1)                               # [B, P, L]
            # x = self.patch_proj(tokens)                                 # [B, P, d_model]

            patches = x_t.unfold(-1, self.patch_len, self.patch_stride)  # [B, C, P, L]
            B, C, P, L = patches.shape
            tokens_flat = patches.reshape(B * C, P, L)                   # [B*C, P, L]
            x_flat = self.patch_proj(tokens_flat)                        # [B*C, P, d_model]
            x = x_flat.reshape(B, C, P, self.d_model).mean(dim=1)       # [B, P, d_model]
        else:
            x = self.input_projection(src) * math.sqrt(self.d_model)   # [B, T, d_model]

        x = self.pos_encoder(x)
        return self.transformer_encoder(x, src_mask)                    # [B, T/P, d_model]

    def forward(
        self,
        src: torch.Tensor,
        src_mask: Optional[torch.Tensor] = None,
        return_embedding: bool = False,
    ):
        n_feat        = self.input_dim - self.output_dim
        x_feat        = src[:, :, :n_feat]
        x_target      = src[:, :, n_feat:]

        x_target_norm = self.revin(x_target, mode='norm')
        src_norm      = torch.cat([x_feat, x_target_norm], dim=-1)

        encoded    = self._encode(src_norm, src_mask)
        last_token = encoded[:, -1, :]
        raw_pred   = self.forecasting_head(last_token)
        pred_norm  = raw_pred.view(raw_pred.shape[0], self.prediction_length, self.output_dim)
        prediction = self.revin(pred_norm, mode='denorm')

        if return_embedding:
            embedding = encoded.mean(dim=1).clone()
            return prediction, embedding
        return prediction

    def get_embedding(self, src: torch.Tensor, src_mask=None) -> torch.Tensor:
        n_feat        = self.input_dim - self.output_dim
        x_target_norm = self.revin(src[:, :, n_feat:], mode='norm')
        src_norm      = torch.cat([src[:, :, :n_feat], x_target_norm], dim=-1)
        encoded       = self._encode(src_norm, src_mask)
        return encoded.mean(dim=1)

    def get_projection(self, src: torch.Tensor, src_mask=None) -> torch.Tensor:
        emb = self.get_embedding(src, src_mask)
        return self.projection_head(emb)


# ---------------------------------------------------------------------------
# Ablation wrapper
# ---------------------------------------------------------------------------

class AblationLLM(nn.Module):
    def __init__(self, base_model: EnhancedTimeSeriesLLM, ablation_config: dict):
        super().__init__()
        self.base_model      = base_model
        self.ablation_config = ablation_config
        self.disabled        = ablation_config.get('disable', [])

    def forward(self, src: torch.Tensor, src_mask=None, return_embedding: bool = False):
        return self.base_model(src, src_mask, return_embedding=return_embedding)

    def get_embedding(self, src: torch.Tensor) -> torch.Tensor:
        if 'tcca' in self.disabled:
            return torch.zeros(src.shape[0], self.base_model.d_model, device=src.device)
        return self.base_model.get_embedding(src)

    def get_projection(self, src: torch.Tensor) -> torch.Tensor:
        if 'cross_modal' in self.disabled or 'tcca' in self.disabled:
            proj_dim = self.base_model.projection_head.net[-1].out_features
            return torch.zeros(src.shape[0], proj_dim, device=src.device)
        return self.base_model.get_projection(src)

Overwriting models/llm_model.py


In [ ]:
#@title losses
%%writefile models/losses.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Optional


def soft_contrastive_loss(z_i: torch.Tensor, z_j: torch.Tensor,
                          temperature: float = 0.07,
                          soft_weight: Optional[torch.Tensor] = None) -> torch.Tensor:
    N = z_i.shape[0]
    z_i = F.normalize(z_i, dim=1)
    z_j = F.normalize(z_j, dim=1)
    z = torch.cat([z_i, z_j], dim=0)
    sim_matrix = torch.matmul(z, z.T) / temperature

    idx = torch.arange(N, device=z.device)
    pos_mask = torch.zeros(2*N, 2*N, dtype=torch.bool, device=z.device)
    pos_mask[idx, idx + N] = True
    pos_mask[idx + N, idx] = True
    diag_mask = torch.eye(2*N, dtype=torch.bool, device=z.device)
    neg_mask = ~(diag_mask | pos_mask)

    pos_logits = sim_matrix[pos_mask].view(2*N, 1)
    neg_logits = sim_matrix[neg_mask].view(2*N, -1)
    all_logits = torch.cat([pos_logits, neg_logits], dim=1)
    labels = torch.zeros(2*N, dtype=torch.long, device=z.device)
    loss = F.cross_entropy(all_logits, labels, reduction='none')

    if soft_weight is not None and soft_weight.shape[0] == N:
        soft_weight = torch.cat([soft_weight, soft_weight], dim=0)
        loss = (loss * soft_weight).mean()
    else:
        loss = loss.mean()
    return loss


def cross_modal_alignment_loss(ts_emb: torch.Tensor, text_emb: torch.Tensor,
                                temperature: float = 0.07) -> torch.Tensor:
    """
    InfoNCE cross-modal loss.
    Both ts_emb and text_emb MUST have the same last dimension before calling.
    SCTA_Loss is responsible for projecting text_emb to match ts_emb dim.
    """
    # Defensive: if dims still mismatch just skip (returns 0 loss)
    if ts_emb.shape[-1] != text_emb.shape[-1]:
        return torch.tensor(0.0, device=ts_emb.device, requires_grad=False)

    ts_emb   = F.normalize(ts_emb,   dim=1)
    text_emb = F.normalize(text_emb, dim=1)
    sim_matrix = torch.matmul(ts_emb, text_emb.T) / temperature
    labels = torch.arange(ts_emb.shape[0], device=ts_emb.device)
    loss_a = F.cross_entropy(sim_matrix,   labels)
    loss_b = F.cross_entropy(sim_matrix.T, labels)
    return (loss_a + loss_b) / 2


class SCTA_Loss(nn.Module):
    """
    Soft-Constrained Temporal Alignment Loss.
    text_proj: lazily-created Linear that projects T_emb [d_model] → [proj_dim]
    so cross_modal_alignment_loss always receives matching dims.
    """
    def __init__(self, lambda_softcl: float = 1.0,
                 lambda_crossmodal: float = 1.0,
                 temperature: float = 0.07,
                 use_softcl: bool = True,
                 use_crossmodal: bool = True):
        super().__init__()
        self.lambda_softcl     = lambda_softcl
        self.lambda_crossmodal = lambda_crossmodal
        self.temperature       = temperature
        self.use_softcl        = use_softcl
        self.use_crossmodal    = use_crossmodal
        self.text_proj         = None   # created lazily on first forward

    def _get_text_proj(self, text_dim: int, ts_dim: int, device) -> nn.Module:
        """Lazily create (or reuse) the text projection layer."""
        if self.text_proj is None or self.text_proj.in_features != text_dim                 or self.text_proj.out_features != ts_dim:
            self.text_proj = nn.Linear(text_dim, ts_dim, bias=False).to(device)
            nn.init.xavier_uniform_(self.text_proj.weight)
        return self.text_proj

    def forward(self, ts_emb_i: torch.Tensor, ts_emb_j: torch.Tensor,
                text_emb: torch.Tensor,
                temporal_constraint_weight: Optional[torch.Tensor] = None
                ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        device = ts_emb_i.device
        softcl_loss    = torch.tensor(0.0, device=device)
        crossmodal_loss = torch.tensor(0.0, device=device)

        if self.use_softcl:
            softcl_loss = soft_contrastive_loss(
                ts_emb_i, ts_emb_j,
                temperature=self.temperature,
                soft_weight=temporal_constraint_weight
            )

        if self.use_crossmodal:
            avg_ts_emb = (ts_emb_i + ts_emb_j) / 2
            ts_dim     = avg_ts_emb.shape[-1]
            text_dim   = text_emb.shape[-1]

            # Project text_emb to ts_dim if dimensions differ
            if text_dim != ts_dim:
                proj = self._get_text_proj(text_dim, ts_dim, device)
                text_emb_aligned = proj(text_emb.to(device))
            else:
                text_emb_aligned = text_emb.to(device)

            crossmodal_loss = cross_modal_alignment_loss(
                avg_ts_emb, text_emb_aligned,
                temperature=self.temperature
            )

        total_loss = (self.lambda_softcl * softcl_loss +
                      self.lambda_crossmodal * crossmodal_loss)
        return total_loss, softcl_loss, crossmodal_loss


class CombinedLoss(nn.Module):
    def __init__(self, lambda_forecast: float = 1.0,
                 lambda_scta: float = 0.3,
                 lambda_spectral: float = 0.05,
                 scta_config: dict = None):
        super().__init__()
        self.lambda_forecast = lambda_forecast
        self.lambda_scta     = lambda_scta
        self.lambda_spectral = lambda_spectral
        self.forecast_loss   = nn.MSELoss()
        if scta_config is None:
            scta_config = {}
        self.scta_loss = SCTA_Loss(**scta_config)

    def forward(self, y_pred, y_true, ts_emb_i, ts_emb_j, text_emb,
                temporal_weight=None):
        forecast_loss = self.forecast_loss(y_pred, y_true)

        # Spectral auxiliary loss — squeeze(-1) safe for both [B,T,1] and [B,T]
        yp = y_pred.squeeze(-1) if y_pred.shape[-1] == 1 else y_pred.mean(-1)
        yt = y_true.squeeze(-1) if y_true.shape[-1] == 1 else y_true.mean(-1)
        spectral_loss = F.mse_loss(
            torch.fft.rfft(yp, dim=-1).abs(),
            torch.fft.rfft(yt, dim=-1).abs()
        )

        scta_total, softcl, crossmodal = self.scta_loss(
            ts_emb_i, ts_emb_j, text_emb, temporal_weight
        )

        total_loss = (self.lambda_forecast * forecast_loss +
                      self.lambda_scta     * scta_total    +
                      self.lambda_spectral * spectral_loss)

        return total_loss, {
            'total':      total_loss.item(),
            'forecast':   forecast_loss.item(),
            'scta':       scta_total.item(),
            'softcl':     softcl.item(),
            'crossmodal': crossmodal.item(),
            'spectral':   spectral_loss.item(),
        }


Overwriting models/losses.py


In [ ]:
#@title helpers.py
%%writefile utils/helpers.py
import torch
import numpy as np
import random
import os
import json
import yaml
from typing import Dict, Any, Optional, List
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# BUG 7 FIXED: os.makedirs(os.path.dirname(filepath)) crashes when filepath
# has no directory component (e.g. 'results.json') because dirname returns ''
# and makedirs('') raises FileNotFoundError. Guard with `if dirpath`.
def save_json(data: Dict, filepath: str):
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=2)


def load_json(filepath: str) -> Dict:
    with open(filepath, 'r') as f:
        return json.load(f)


def save_yaml(data: Dict, filepath: str):
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)


def load_yaml(filepath: str) -> Dict:
    with open(filepath, 'r') as f:
        return yaml.safe_load(f)


def get_device(device_str: str = 'cuda') -> torch.device:
    if device_str == 'cuda' and torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def print_model_summary(model: torch.nn.Module, input_size: tuple = None):
    print("\n" + "="*80)
    print("Model Architecture")
    print("="*80)
    print(model)
    print("="*80)
    total_params = count_parameters(model)
    print(f"\nTotal trainable parameters: {total_params:,}")
    print(f"Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB (assuming fp32)")
    print("="*80 + "\n")


def create_exp_directory(base_dir: str = 'experiments',
                         exp_name: Optional[str] = None) -> str:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_name = f'{exp_name}_{timestamp}' if exp_name else f'exp_{timestamp}'
    exp_dir = os.path.join(base_dir, exp_name)
    os.makedirs(exp_dir, exist_ok=True)
    for sub in ('checkpoints', 'logs', 'plots'):
        os.makedirs(os.path.join(exp_dir, sub), exist_ok=True)
    return exp_dir


def plot_training_curves(train_losses, val_losses,
                         save_path=None, title='Training Curves'):
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    plt.plot(epochs, val_losses,   'r-', label='Validation Loss', linewidth=2)
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(title)
    plt.legend(); plt.grid(True, alpha=0.3)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    else:
        plt.show()
    plt.close()


def plot_predictions(y_true, y_pred, save_path=None,
                     title='Predictions vs Ground Truth', num_samples=100):
    if len(y_true) > num_samples:
        indices = np.random.choice(len(y_true), num_samples, replace=False)
        y_true, y_pred = y_true[indices], y_pred[indices]
    if len(y_true.shape) > 1 and y_true.shape[1] > 1:
        y_true, y_pred = y_true[:, 0], y_pred[:, 0]

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    x = np.arange(len(y_true))
    plt.plot(x, y_true, 'b-', label='Ground Truth', linewidth=2, alpha=0.7)
    plt.plot(x, y_pred, 'r--', label='Predictions',  linewidth=2, alpha=0.7)
    plt.xlabel('Sample'); plt.ylabel('Value')
    plt.title('Time Series Comparison'); plt.legend(); plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.scatter(y_true, y_pred, alpha=0.5, s=20)
    mn = min(y_true.min(), y_pred.min()); mx = max(y_true.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect')
    plt.xlabel('Ground Truth'); plt.ylabel('Predictions')
    plt.title('Scatter Plot'); plt.legend(); plt.grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=16, y=1.02); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    else:
        plt.show()
    plt.close()


def plot_error_distribution(y_true, y_pred, save_path=None, title='Error Distribution'):
    errors = y_true.flatten() - y_pred.flatten()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(errors, bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[0].set_xlabel('Error'); axes[0].set_ylabel('Frequency')
    axes[0].set_title('Error Histogram'); axes[0].grid(True, alpha=0.3)
    axes[1].boxplot(errors, vert=True)
    axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
    axes[1].set_ylabel('Error'); axes[1].set_title('Error Box Plot')
    axes[1].grid(True, alpha=0.3)
    plt.suptitle(title, fontsize=16, y=1.02); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    else:
        plt.show()
    plt.close()


def format_time(seconds: float) -> str:
    h, rem = divmod(int(seconds), 3600)
    m, s   = divmod(rem, 60)
    if h:   return f"{h}h {m}m {s}s"
    if m:   return f"{m}m {s}s"
    return  f"{s}s"


def calculate_dtw_distance(ts1: np.ndarray, ts2: np.ndarray) -> float:
    n, m = len(ts1), len(ts2)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(ts1[i-1] - ts2[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j], dtw_matrix[i, j-1], dtw_matrix[i-1, j-1]
            )
    return dtw_matrix[n, m]


def generate_text_description(ts: np.ndarray) -> str:
    mean_val = np.mean(ts); std_val = np.std(ts)
    trend = 'increasing' if ts[-1] > ts[0] else 'decreasing'
    volatility = 'high' if std_val > mean_val * 0.3 else 'low'
    return (f"A {trend} time series with {volatility} volatility. "
            f"Mean value is {mean_val:.2f} with standard deviation {std_val:.2f}.")


class Logger:
    def __init__(self, log_file: Optional[str] = None):
        self.log_file = log_file
        if log_file:
            dirpath = os.path.dirname(log_file)
            if dirpath:
                os.makedirs(dirpath, exist_ok=True)

    def log(self, message: str, print_msg: bool = True):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        log_msg = f"[{timestamp}] {message}"
        if print_msg:
            print(log_msg)
        if self.log_file:
            with open(self.log_file, 'a') as f:
                f.write(log_msg + '\n')


# BUG 6 FIXED: save_checkpoint was imported by train.py but did not exist.
# create_checkpoint only built the dict; save_checkpoint builds + writes to disk.
def save_checkpoint(model: torch.nn.Module,
                    optimizer: torch.optim.Optimizer,
                    epoch: int,
                    loss: float,
                    filepath: str,
                    config: Optional[Dict] = None,
                    additional_info: Optional[Dict] = None) -> Dict:
    """Build a checkpoint dict and save it to `filepath`."""
    checkpoint = {
        'epoch':               epoch,
        'model_state_dict':    model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss':                loss,
        'config':              config,
        'timestamp':           datetime.now().isoformat(),
    }
    if additional_info:
        checkpoint.update(additional_info)

    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    torch.save(checkpoint, filepath)
    return checkpoint


def create_checkpoint(model: torch.nn.Module,
                      optimizer: torch.optim.Optimizer,
                      epoch: int,
                      loss: float,
                      config: Dict,
                      additional_info: Optional[Dict] = None) -> Dict:
    """Build a checkpoint dict without saving (legacy helper)."""
    checkpoint = {
        'epoch':               epoch,
        'model_state_dict':    model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss':                loss,
        'config':              config,
        'timestamp':           datetime.now().isoformat(),
    }
    if additional_info:
        checkpoint.update(additional_info)
    return checkpoint


def load_checkpoint(filepath: str,
                    model: torch.nn.Module,
                    optimizer: Optional[torch.optim.Optimizer] = None,
                    device: torch.device = torch.device('cpu')) -> Dict:
    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint

Overwriting utils/helpers.py


In [ ]:
#@title metrics.py
%%writefile utils/metrics.py
import torch
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from typing import Union


# ---------------------------------------------------------------------------
# Basic metrics
# ---------------------------------------------------------------------------

def calculate_mse(y_true, y_pred) -> float:
    if isinstance(y_true, torch.Tensor): y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor): y_pred = y_pred.cpu().numpy()
    return mean_squared_error(y_true.flatten(), y_pred.flatten())


def calculate_mae(y_true, y_pred) -> float:
    if isinstance(y_true, torch.Tensor): y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor): y_pred = y_pred.cpu().numpy()
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())


def calculate_rmse(y_true: Union[np.ndarray, torch.Tensor],
                   y_pred: Union[np.ndarray, torch.Tensor]) -> float:
    return np.sqrt(calculate_mse(y_true, y_pred))


# ---------------------------------------------------------------------------
# Hallucination Rate — fixed: 3σ threshold
# ---------------------------------------------------------------------------

def calculate_hallucination_rate(
    y_true: Union[np.ndarray, torch.Tensor],
    y_pred: Union[np.ndarray, torch.Tensor],
    threshold: float = 3.0,          # ← was 2.0; see module docstring
) -> float:
    """
    Fraction of predictions whose absolute error is a statistical outlier,
    expressed as a percentage.

    A z-score threshold of 3.0 is used (3σ rule): for a perfectly Gaussian
    error distribution this yields ~0.3% HR, which is actually discriminative.
    The original threshold of 2σ produces ~5% HR for any unbiased model,
    making it impossible to distinguish good models from bad ones.

    Args:
        y_true:    Ground truth values.
        y_pred:    Predicted values.
        threshold: Z-score cutoff (default 3.0).

    Returns:
        Hallucination rate as a percentage in [0, 100].
    """
    if isinstance(y_true, torch.Tensor): y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor): y_pred = y_pred.cpu().numpy()

    errors   = np.abs(y_true.flatten() - y_pred.flatten())
    mean_err = np.mean(errors)
    std_err  = np.std(errors) + 1e-8
    z_scores = (errors - mean_err) / std_err

    return float(np.sum(z_scores > threshold) / len(errors) * 100)


# ---------------------------------------------------------------------------
# Perplexity — fixed: residual variance as σ²
# ---------------------------------------------------------------------------

def calculate_perplexity(
    y_true: Union[np.ndarray, torch.Tensor],
    y_pred: Union[np.ndarray, torch.Tensor],
) -> float:
    """
    Perplexity of the forecast under an assumed Gaussian error model.

    σ² is estimated as the *variance* of the residuals (not raw MSE).
    MSE equals variance only when the mean residual is zero; using MSE
    instead inflates perplexity for any model with systematic bias.

    Formula:
        σ²   = Var(y_true - y_pred)
        NLL  = 0.5 * log(2π σ²) + 0.5          [mean term of Gaussian NLL]
        PPL  = exp(NLL)

    Returns:
        Perplexity value (≥ 1; lower is better).
    """
    if isinstance(y_true, torch.Tensor): y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor): y_pred = y_pred.cpu().numpy()

    residuals    = y_true.flatten() - y_pred.flatten()
    sigma2       = np.var(residuals) + 1e-8          # ← was mean_squared_error

    # Mean negative log-likelihood per point under N(0, σ²)
    nll          = 0.5 * np.log(2 * np.pi * sigma2) + 0.5
    return float(np.exp(nll))


# ---------------------------------------------------------------------------
# MASE — new: scale-free metric reported by baselines
# ---------------------------------------------------------------------------

def calculate_mase(
    y_true: Union[np.ndarray, torch.Tensor],
    y_pred: Union[np.ndarray, torch.Tensor],
    seasonality: int = 1,
) -> float:
    """
    Mean Absolute Scaled Error (Hyndman & Koehler, 2006).

    Normalises MAE by the mean absolute error of the naïve seasonal forecast
    on the *true* series, making the metric scale-free and comparable across
    datasets with different units.

    MASE = MAE(model) / MAE(naïve)

    where MAE(naïve) = mean |y[t] - y[t - seasonality]| over all t.

    Values:
        MASE < 1  → better than the naïve seasonal forecast (good)
        MASE = 1  → equal to naïve
        MASE > 1  → worse than naïve

    Args:
        y_true:      Ground truth [N] or [N, T] or [N, T, D].
        y_pred:      Predictions — same shape as y_true.
        seasonality: Lag used for the naïve baseline (default 1 = random walk).
                     Use 24 for hourly data with daily seasonality, etc.

    Returns:
        MASE value (float). Returns NaN if the naïve MAE is zero (flat series).
    """
    if isinstance(y_true, torch.Tensor): y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor): y_pred = y_pred.cpu().numpy()

    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()

    mae_model = np.mean(np.abs(y_true_flat - y_pred_flat))

    # Naïve seasonal forecast error on the true series
    if len(y_true_flat) <= seasonality:
        # Not enough points to compute the naive error — return NaN gracefully
        return float('nan')

    naive_errors = np.abs(y_true_flat[seasonality:] - y_true_flat[:-seasonality])
    mae_naive    = np.mean(naive_errors)

    if mae_naive < 1e-8:
        # Flat ground-truth series: naive MAE ≈ 0, MASE undefined
        return float('nan')

    return float(mae_model / mae_naive)


# ---------------------------------------------------------------------------
# Temporal Coherence Error (unchanged)
# ---------------------------------------------------------------------------

def calculate_tce(
    y_true: Union[np.ndarray, torch.Tensor],
    y_pred: Union[np.ndarray, torch.Tensor],
) -> float:
    """
    Temporal Coherence Error — MAE of consecutive first-order differences.
    Measures how well predictions preserve temporal dynamics.
    """
    if isinstance(y_true, torch.Tensor): y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor): y_pred = y_pred.cpu().numpy()

    y_true = y_true.reshape(len(y_true), -1)
    y_pred = y_pred.reshape(len(y_pred), -1)

    diff_true = np.diff(y_true, axis=0)
    diff_pred = np.diff(y_pred, axis=0)

    return float(np.mean(np.abs(diff_true - diff_pred)))


# ---------------------------------------------------------------------------
# Soft-DTW (unchanged)
# ---------------------------------------------------------------------------

def calculate_soft_dtw(
    y_true: Union[np.ndarray, torch.Tensor],
    y_pred: Union[np.ndarray, torch.Tensor],
    gamma: float = 1.0,
    max_samples: int = 64,
) -> float:
    """
    Mean Soft-DTW distance between predicted and true sequences.
    Averaged over up to `max_samples` sequences for speed.
    """
    if isinstance(y_true, torch.Tensor): y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor): y_pred = y_pred.cpu().numpy()

    if y_true.ndim == 1:
        y_true = y_true.reshape(1, -1)
        y_pred = y_pred.reshape(1, -1)
    elif y_true.ndim > 2:
        y_true = y_true.reshape(len(y_true), -1)[:, :y_true.shape[1]]
        y_pred = y_pred.reshape(len(y_pred), -1)[:, :y_pred.shape[1]]

    n       = min(len(y_true), max_samples)
    indices = (np.random.choice(len(y_true), n, replace=False)
               if len(y_true) > n else np.arange(n))

    return float(np.mean([_soft_dtw_1d(y_true[i], y_pred[i], gamma) for i in indices]))


def _soft_dtw_1d(ts1: np.ndarray, ts2: np.ndarray, gamma: float = 1.0) -> float:
    n, m = len(ts1), len(ts2)
    R    = np.full((n + 1, m + 1), np.inf)
    R[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost     = (ts1[i - 1] - ts2[j - 1]) ** 2
            vals     = np.array([R[i-1, j], R[i, j-1], R[i-1, j-1]])
            soft_min = -gamma * np.log(np.sum(np.exp(-vals / (gamma + 1e-8))) + 1e-8)
            R[i, j]  = cost + soft_min
    return float(R[n, m])


# ---------------------------------------------------------------------------
# Evaluation runner
# ---------------------------------------------------------------------------

def evaluate_model(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
    scaler=None,
    mase_seasonality: int = 1,
) -> tuple:
    """
    Run model over dataloader and return all metrics.

    Args:
        model:            Trained model (eval mode set internally).
        dataloader:       Test/val DataLoader.
        device:           torch.device.
        scaler:           Optional inverse-transform scaler (StandardScaler).
        mase_seasonality: Seasonality lag for MASE (1 = random-walk baseline).

    Returns:
        (metrics dict, y_pred array, y_true array)
    """
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch in dataloader:
            X_i, _, _, _, y = batch
            y_pred = model(X_i.to(device))
            all_preds.append(y_pred.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds,   axis=0)
    y_true = np.concatenate(all_targets, axis=0)

    # Inverse-transform if scaler provided (StandardScaler from preprocessing)
    # Snapshot normalized arrays for baseline-comparable metrics
    y_pred_norm = y_pred.copy()
    y_true_norm = y_true.copy()

    if scaler is not None:
        orig_shape = y_pred.shape                               # [N, T, D]
        y_pred = scaler.inverse_transform(
            y_pred.reshape(-1, orig_shape[-1])).reshape(orig_shape)
        y_true = scaler.inverse_transform(
            y_true.reshape(-1, orig_shape[-1])).reshape(orig_shape)

    metrics = {
        'MSE':        calculate_mse(y_true, y_pred),
        'MAE':        calculate_mae(y_true, y_pred),
        'RMSE':       calculate_rmse(y_true, y_pred),
        'MASE':       calculate_mase(y_true, y_pred, seasonality=mase_seasonality),
        # HR and Perplexity on normalized space — same scale as baselines,
        # and unaffected by inverse-transform magnitude differences
        'HR':         calculate_hallucination_rate(y_pred_norm, y_true_norm),
        'Perplexity': calculate_perplexity(y_true_norm, y_pred_norm),
        'TCE':        calculate_tce(y_true, y_pred),
        'Soft-DTW':   calculate_soft_dtw(y_true, y_pred),
    }

    # Normalized metrics (z-score space) — comparable to PatchTST/TimeLLM baselines
    metrics['MSE_norm'] = float(np.mean((y_pred_norm - y_true_norm) ** 2))
    metrics['MAE_norm'] = float(np.mean(np.abs(y_pred_norm - y_true_norm)))

    return metrics, y_pred, y_true


# ---------------------------------------------------------------------------
# Pretty printer
# ---------------------------------------------------------------------------

def print_metrics(metrics: dict, prefix: str = "") -> None:
    mase_str = (f"{metrics['MASE']:.6f}"
                if not np.isnan(metrics.get('MASE', float('nan')))
                else "n/a (flat series)")
    print(f"\n{prefix}Evaluation Metrics:")
    print("-" * 50)
    print(f"MSE:          {metrics['MSE']:.6f}")
    print(f"MAE:          {metrics['MAE']:.6f}")
    print(f"RMSE:         {metrics['RMSE']:.6f}")
    print(f"MASE:         {mase_str}")
    print(f"HR (%):       {metrics['HR']:.2f}   [3σ threshold]")
    print(f"Perplexity:   {metrics['Perplexity']:.6f}")
    print(f"TCE:          {metrics['TCE']:.6f}")
    print(f"Soft-DTW:     {metrics['Soft-DTW']:.6f}")
    if 'MSE_norm' in metrics:
        print(f"MSE_norm:     {metrics['MSE_norm']:.6f}   [normalized — baseline-comparable]")
        print(f"MAE_norm:     {metrics['MAE_norm']:.6f}   [normalized — baseline-comparable]")
    print("-" * 50)


Overwriting utils/metrics.py


##init

In [ ]:
!touch config/__init__.py

In [ ]:
!touch experiments/__init__.py


In [ ]:
%%writefile models/__init__.py
from .llm_model import EnhancedTimeSeriesLLM, AblationLLM
from .losses import SCTA_Loss, CombinedLoss
from .hsqp import HSQP

__all__ = ['EnhancedTimeSeriesLLM', 'AblationLLM', 'SCTA_Loss', 'CombinedLoss', 'HSQP']


Overwriting models/__init__.py


In [ ]:
%%writefile data/__init__.py
from .preprocessing import TimeSeriesPreprocessor
from .dataset import TimeSeriesDataset, AblationDataset

__all__ = ['TimeSeriesPreprocessor', 'TimeSeriesDataset', 'AblationDataset']



Overwriting data/__init__.py


In [ ]:
%%writefile utils/__init__.py

from .metrics import (
    calculate_mse, calculate_mae, calculate_rmse,
    calculate_hallucination_rate, calculate_perplexity,
    calculate_tce, calculate_soft_dtw,
    evaluate_model, print_metrics,
)

__all__ = [
    'calculate_mse', 'calculate_mae', 'calculate_rmse',
    'calculate_hallucination_rate', 'calculate_perplexity',
    'calculate_tce', 'calculate_soft_dtw',
    'evaluate_model', 'print_metrics',
]

Overwriting utils/__init__.py


##Readme

In [ ]:
#@title README
%%writefile README.md

# Time Series LLM — Project README

A transformer-based time series forecasting pipeline with contrastive learning (SCTA),
hierarchical patch quantization (HSQP), and a full ablation framework.

---

## Project Structure

```
timeseries/
├── config/
│   ├── __init__.py
│   └── config.yaml            # All hyperparameters and dataset paths
├── data/
│   ├── __init__.py
│   ├── dataset.py             # TimeSeriesDataset, AblationDataset
│   └── preprocessing.py       # TimeSeriesPreprocessor
├── experiments/
│   ├── __init__.py
│   └── train.py               # Trainer class, train_on_dataset()
├── models/
│   ├── __init__.py
│   ├── encoder.py             # TimeSeriesEncoder, TemporalConvNet
│   ├── hsqp.py                # HSQP patching and quantization
│   ├── llm_model.py           # EnhancedTimeSeriesLLM, AblationLLM
│   └── losses.py              # SoftCL, CrossModal, SCTA, CombinedLoss
├── utils/
│   ├── __init__.py
│   ├── helpers.py             # Device, checkpointing, plotting utilities
│   └── metrics.py             # MSE, MAE, RMSE, HR, Perplexity, TCE, Soft-DTW
├── ablation.py                # Ablation study runner
├── evaluate.py                # Standalone evaluation script
├── main.py                    # Main training entry point
└── requirements.txt
```

---

## 1. Setup

### 1.1 Create directory structure and download data

```python
#@title Step 1 — Create folders
!mkdir timeseries
%cd timeseries
!mkdir config data experiments models utils
```

```python
#@title Step 2 — Download data from Google Drive
!gdown https://drive.google.com/drive/folders/1jJF4eyd6jOhtDdBxsBqZxADycebXLKi8?usp=sharing \
    -O . --folder
```

```python
#@title Step 3 — Install dependencies
!pip install -r requirements.txt
```

### 1.2 Verify GPU is available

```python
#@title GPU Check
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
```

> **Important:** Make sure your Colab runtime is set to **GPU** before running.
> Runtime → Change runtime type → Hardware accelerator → GPU

### 1.3 Key settings in `config/config.yaml`

| Parameter | Default | Recommended (GPU) | Notes |
|---|---|---|---|
| `device` | `cuda` | `cuda` | Change to `cpu` only if no GPU |
| `training.batch_size` | `16` | `128` | Larger = better GPU utilisation |
| `training.epochs` | `20` | `50` | More epochs for larger models |
| `training.learning_rate` | `0.0001` | `0.0001` | Adam default works well |
| `model.llm.d_model` | `64` | `256` | Increase for GPU saturation |
| `model.llm.n_layers` | `4` | `6` | Depth vs. speed trade-off |
| `main.py MAX_TRAIN_SAMPLES` | `5000` | `50000` | Hard-coded cap — edit directly |

---

## 2. Training

### 2.1 Train on all configured datasets

```python
#@title Train — all datasets (uses config.yaml)
!python main.py
```

### 2.2 Train on a single dataset

```python
#@title Train — single dataset
!python main.py --dataset ETTm1
```

### 2.3 Train on multiple specific datasets

```python
#@title Train — multiple datasets
!python main.py --datasets ETTh1 ETTh2 ETTm1 ETTm2
```

### 2.4 Override hyperparameters at runtime

```python
#@title Train — custom hyperparameters
!python main.py \
    --dataset ETTm1 \
    --epochs 50 \
    --batch_size 128 \
    --lr 0.0001 \
    --device cuda
```

### 2.5 All available training flags

| Flag | Type | Default | Description |
|---|---|---|---|
| `--config` | str | `config/config.yaml` | Path to config file |
| `--datasets` | list | all in config | Space-separated dataset names |
| `--device` | str | from config | `cuda` or `cpu` |
| `--epochs` | int | from config | Number of training epochs |
| `--batch_size` | int | from config | Batch size |
| `--lr` | float | from config | Learning rate |
| `--max_samples` | int | `5000` | Max training samples per dataset |
| `--max_input_dim` | int | `200` | Skip datasets exceeding this feature count |

### 2.6 Supported datasets

| Name | Description | Target | Frequency |
|---|---|---|---|
| `timeseries` | Custom dataset | WS, RH, T | Daily |
| `ETTh1` | Electricity Transformer Temp (hourly) | OT | 1h |
| `ETTh2` | Electricity Transformer Temp (hourly) | OT | 1h |
| `ETTm1` | Electricity Transformer Temp (minutely) | OT | 15min |
| `ETTm2` | Electricity Transformer Temp (minutely) | OT | 15min |
| `traffic` | Road traffic (high-dim) | OT | 1h |
| `electricity` | Electricity consumption (high-dim) | OT | 1h |
| `weather` | Weather station data | OT | 10min |
| `national_illness` | US flu/illness surveillance | OT | Weekly |

> **Note:** `traffic` and `electricity` have 800+ features. They are skipped by default
> when `--max_input_dim 200`. Set `--max_input_dim 1000` to include them.

### 2.7 Training outputs

After training, the following are saved automatically:

```
saved_models/
    best_model_<dataset>.pth       # Best checkpoint by validation loss
    scaler_<dataset>.pkl           # MinMaxScaler for inverse transform
results/
    results_<dataset>.json         # Metrics + training history
logs/
```

---

## 3. Ablation Studies

Ablation studies systematically disable individual components to measure their contribution.

### 3.1 Run all ablations on all datasets

```python
#@title Ablation — full study
!python ablation.py
```

### 3.2 Run all ablations on specific datasets

```python
#@title Ablation — specific datasets
!python ablation.py --datasets ETTm1 ETTh1
```

### 3.3 Run a specific ablation variant

```python
#@title Ablation — single variant
!python ablation.py --datasets ETTm1 --ablations w/o_TCCA
```

### 3.4 Run multiple specific ablation variants

```python
#@title Ablation — multiple variants
!python ablation.py \
    --datasets ETTm1 \
    --ablations full_model w/o_TCCA w/o_Soft-DTW w/o_Cross-Modal
```

### 3.5 Available ablation variants

| Name | Disabled Component | What it tests |
|---|---|---|
| `full_model` | nothing | Baseline — all components active |
| `w/o_TCCA` | `tcca` | Effect of temporal contrastive alignment |
| `w/o_Soft-DTW` | `soft_dtw` | Effect of soft-DTW temporal weighting |
| `w/o_Cross-Modal` | `cross_modal` | Effect of text-series alignment |
| `w/o_SQT` | `sqt` | Effect of subspace quantization tokens |
| `w/o_Quantization` | `quantization` | Effect of HSQP bit-width quantization |

### 3.6 All available ablation flags

| Flag | Type | Default | Description |
|---|---|---|---|
| `--config` | str | `config/config.yaml` | Config path |
| `--datasets` | list | all in config | Datasets to ablate on |
| `--ablations` | list | all variants | Ablation names to run |
| `--device` | str | from config | `cuda` or `cpu` |
| `--epochs` | int | from config | Epochs per ablation run |
| `--batch_size` | int | from config | Batch size |
| `--lr` | float | from config | Learning rate |
| `--max_samples` | int | `5000` | Max training samples |
| `--max_input_dim` | int | `200` | Feature dimension cap |

### 3.7 Ablation outputs

```
saved_models/
    best_model_<dataset>_<ablation_name>.pth
results/
    ablation_comparison_<dataset>.csv    # Side-by-side metric table
    ablation_results_all.json            # Full results (all datasets × all ablations)
```

---

## 4. Inference / Evaluation

### 4.1 Evaluate a trained model on its test set

```python
#@title Evaluate — test set
!python evaluate.py --dataset ETTm1
```

### 4.2 Evaluate on a different split

```python
#@title Evaluate — validation set
!python evaluate.py --dataset ETTm1 --split val
```

### 4.3 Evaluate with saved plots

```python
#@title Evaluate — with prediction plots
!python evaluate.py \
    --dataset ETTm1 \
    --split test \
    --save_plots \
    --output_dir evaluation_results/
```

### 4.4 Evaluate a specific checkpoint

```python
#@title Evaluate — custom checkpoint
!python evaluate.py \
    --dataset ETTm1 \
    --checkpoint saved_models/best_model_ETTm1.pth \
    --save_plots
```

### 4.5 All available evaluation flags

| Flag | Type | Default | Description |
|---|---|---|---|
| `--config` | str | `config/config.yaml` | Config path |
| `--dataset` | str | **required** | Dataset name to evaluate |
| `--checkpoint` | str | auto | Path to `.pth` file; auto-resolves if omitted |
| `--split` | str | `test` | `train`, `val`, or `test` |
| `--batch_size` | int | `32` | Evaluation batch size |
| `--save_plots` | flag | off | Save prediction and error plots |
| `--output_dir` | str | `evaluation_results` | Where to save plots and JSON |

### 4.6 Evaluation metrics explained

| Metric | Description |
|---|---|
| **MSE** | Mean Squared Error — penalises large deviations heavily |
| **MAE** | Mean Absolute Error — robust to outliers |
| **RMSE** | Root MSE — same units as target |
| **HR** | Hallucination Rate — % predictions with z-score error > 2σ |
| **Perplexity** | Gaussian NLL exponentiated — model confidence proxy |
| **TCE** | Temporal Coherence Error — MAE of first-order differences |
| **Soft-DTW** | Differentiable Dynamic Time Warping similarity |

### 4.7 Programmatic inference (in a notebook)

```python
#@title Inference — programmatic
import torch, yaml
from models.llm_model import EnhancedTimeSeriesLLM

# Load config
with open('config/config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instantiate model
model = EnhancedTimeSeriesLLM(
    input_dim=7,           # match your dataset
    output_dim=1,
    prediction_length=config['data']['prediction_length'],
    **config['model']['llm']
).to(device)

# Load checkpoint
ckpt = torch.load('saved_models/best_model_ETTm1.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Run inference  (X shape: [batch, seq_len, features])
X = torch.randn(1, 96, 7).to(device)
with torch.no_grad():
    prediction = model(X)   # shape: [1, prediction_length, output_dim]

print("Prediction shape:", prediction.shape)
print("Values:", prediction.cpu().numpy())
```

---

## 5. Quick-Start Recipes

### Full pipeline in one go (GPU-optimised)

```python
#@title Full pipeline — GPU optimised
# 1. Train
!python main.py --dataset ETTm1 --epochs 50 --batch_size 128

# 2. Ablation
!python ablation.py --datasets ETTm1 --batch_size 128

# 3. Evaluate with plots
!python evaluate.py --dataset ETTm1 --save_plots
```

### Reproduce results on all ETT benchmarks

```python
#@title All ETT benchmarks
!python main.py \
    --datasets ETTh1 ETTh2 ETTm1 ETTm2 \
    --epochs 50 \
    --batch_size 128 \
    --max_samples 50000
```

---

## 6. Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| GPU RAM ~0.2 GB, compute ~0% | Model too small / batch too small | Increase `d_model`, `batch_size` |
| `CUDA not available` warning | Colab runtime is CPU | Runtime → Change runtime type → GPU |
| `FileNotFoundError: config.yaml` | Wrong working directory | `%cd timeseries` before running |
| `KeyError: 'scaler'` in ablation | Preprocessor key mismatch | Ensure `preprocessing.py` saves `scaler` in `data_dict` |
| Very fast training, poor metrics | `MAX_TRAIN_SAMPLES=5000` cap | Edit `main.py` or pass `--max_samples 50000` |
| Ablation checkpoint not found | Name contains `/` (e.g. `w/o_TCCA`) | Already handled — `/` is replaced with `_` in filenames |

Overwriting README.md


In [ ]:
#@title DESIGN_DECISIONS
%%writefile DESIGN_DECISIONS.md

# Design Decisions — Reasoning & Justification

This document explains the reasoning behind every significant architectural,
algorithmic, and engineering decision in this pipeline.

---

## 1. Core Architecture: Transformer Encoder for Time Series

**Decision:** Use a standard `TransformerEncoder` (`nn.TransformerEncoderLayer`) as the
backbone rather than an RNN/LSTM or a full seq2seq decoder.

**Reasoning:**

Traditional RNNs and LSTMs process time series sequentially — each hidden state depends
on the previous one. This creates two problems: vanishing gradients over long sequences
and an inability to parallelise computation during training. Transformers solve both by
computing all-pairs attention in a single matrix multiply, making them natively parallel
and capable of capturing long-range dependencies (e.g., weekly seasonality in hourly data)
without information decay.

The encoder-only design (no decoder) is used because the forecasting head reads from
the last token of the encoder output. This is analogous to BERT-style "pooling from
the [CLS] token", and it is simpler and faster than seq2seq decoding while being
sufficient for fixed-horizon regression (predicting `prediction_length` steps ahead).

`batch_first=True` is set on `TransformerEncoderLayer` specifically to avoid an
unnecessary transpose at every forward pass — a small but consistent efficiency gain.

---

## 2. Input Projection + √d_model Scaling

**Decision:** Project raw features into `d_model` dimensions with a linear layer,
then scale by `√d_model` before the positional encoder.

**Reasoning:**

Raw input features (temperature, load, wind speed, etc.) can have arbitrary scales.
Projecting to a fixed `d_model` dimension normalises representation capacity —
each feature dimension contributes equally to attention computation regardless
of the number of original input channels.

The `× √d_model` scaling is taken directly from "Attention Is All You Need"
(Vaswani et al., 2017). Without it, the dot-product attention scores grow with `d_model`,
pushing the softmax into near-zero-gradient saturation regions. Scaling by `√d_model`
keeps scores in a numerically stable range before softmax.

---

## 3. Sinusoidal Positional Encoding (not learned)

**Decision:** Use fixed sinusoidal positional encoding instead of learned positional
embeddings.

**Reasoning:**

Learned positional embeddings require the model to see every possible sequence position
during training. For time series that arrive in variable-length windows (e.g., train on
`seq_length=96` but later evaluate on a different window), learned embeddings
generalise poorly out of range.

Sinusoidal encodings are deterministic and extend to arbitrary sequence lengths.
They also embed relative position implicitly: `sin(pos)/cos(pos)` relationships allow
the model to compute `pos_a - pos_b` via linear combinations, helping it attend to
patterns like "4 steps ago" without ever seeing that exact position during training.

---

## 4. GELU Activation (not ReLU)

**Decision:** Use `activation='gelu'` in the Transformer feed-forward sub-layers.

**Reasoning:**

ReLU hard-zeros negative activations, creating sparse, potentially dead neurons
during training. GELU (Gaussian Error Linear Unit) is a smooth approximation:
it down-weights small negative inputs rather than zeroing them, preserving more
gradient signal. Empirically, GELU outperforms ReLU on most language and time
series transformer benchmarks (GPT-2, BERT, PatchTST all use GELU).

---

## 5. Multi-Step Forecasting via Linear Head (not autoregressive)

**Decision:** Predict all `prediction_length` steps simultaneously with a single
`Linear(d_model, output_dim × prediction_length)` layer, then reshape.

**Reasoning:**

Autoregressive decoding (predicting step t+1, feeding it back to predict t+2, etc.)
accumulates errors multiplicatively — a bad prediction at t+1 corrupts every
subsequent step. This is called *error propagation* and is a well-known weakness
of RNN decoders.

Direct multi-output prediction from the encoder's last token avoids this completely.
The model learns to represent all future information in a single latent state.
This matches the "Direct Multi-Output" strategy shown to outperform autoregressive
approaches in PatchTST and Informer benchmarks on standard datasets.

The reshape (`view(batch, prediction_length, output_dim)`) is purely cosmetic —
the linear layer already computes all future values jointly.

---

## 6. HSQP — Hierarchical Subspace Quantization Patching

**Decision:** Implement patch-based tokenisation with subspace quantization (HSQP)
for preprocessing.

**Reasoning:**

Raw time series fed token-by-token (one timestep = one token) create sequences of
length 96+ that are computationally expensive for attention (O(L²) complexity) and
semantically poor — individual timesteps carry little meaning on their own.

Patching (grouping L timesteps into a single token) reduces sequence length by a
factor of `patch_length / stride` while preserving local shape information. This is
exactly the motivation behind PatchTST (2023) and TS2Vec. In this project:
- `patch_length=24` captures one day's worth of hourly data per token
- `stride=12` creates 50% overlap between adjacent patches, preventing information
  loss at patch boundaries

Subspace quantization (K-Means clustering of patches into `k=10` codewords, stored
at 8-bit precision with affine quantization) provides two benefits:
1. Compression — reduces storage and transfer cost for long series
2. Discrete symbolic representation — facilitates alignment with discrete text tokens
   in the cross-modal loss (see Section 9)

The `tol=0.01` and `alpha=0.01` K-Means parameters balance convergence speed
against codebook quality.

---

## 7. MinMaxScaler (not StandardScaler or RobustScaler)

**Decision:** Normalise all input features with `MinMaxScaler` to [0, 1].

**Reasoning:**

Time series forecasting benefits from bounded input ranges because:
1. Transformer attention weights are computed via softmax, which is sensitive to
   the magnitude of query-key dot products. Unbounded inputs (from StandardScaler
   with outlier-heavy data) can produce very large dot products, saturating softmax.
2. MinMax-scaled targets also make the MSE loss directly interpretable as a fraction
   of the total value range, which aids hyperparameter tuning.

StandardScaler is commonly used in regression but produces unbounded outputs when
data has heavy tails (e.g., traffic spikes, illness surges). RobustScaler is better
for outlier-heavy data but complicates inverse transform during evaluation.

MinMax is retained as a pragmatic default; a user dealing with extreme outliers
should switch to RobustScaler and update `preprocessing.py`.

---

## 8. SoftCL — Soft Contrastive Loss

**Decision:** Train with soft contrastive loss (SoftCL) using noise-augmented views
rather than a purely supervised MSE loss.

**Reasoning:**

Supervised-only training (minimising MSE on `y_pred` vs `y_true`) teaches the model
to map inputs to outputs but gives no explicit signal about representation quality —
two different time series with similar shapes could receive arbitrarily different
embeddings.

Contrastive learning (SimCLR-style) enforces that augmented views of the same series
produce similar embeddings while different series produce dissimilar ones. This builds
a smoother, more structured latent space, which regularises the forecasting head and
helps generalisation across datasets.

The "soft" variant adds a `temporal_weight` to each positive pair, down-weighting
pairs that are temporally distant from each other. This is important for time series
because two windows from the same series can be semantically dissimilar if they come
from different seasons or trends. Hard contrastive loss would incorrectly penalise
the model for separating them; the soft weight resolves this.

Augmentation is minimal (Gaussian noise, σ=0.01) to preserve the temporal structure
of the series. Stronger augmentations (masking, permutation) would distort the
seasonal and trend patterns the model needs to learn.

---

## 9. Cross-Modal Alignment Loss (InfoNCE)

**Decision:** Align time series embeddings with text embeddings using a bidirectional
InfoNCE loss.

**Reasoning:**

The eventual goal of LLM-augmented forecasting is to leverage textual knowledge
(e.g., "electricity demand drops on public holidays") to improve predictions.
The cross-modal loss trains the time series encoder to produce embeddings that
are geometrically close to semantically related text representations.

InfoNCE (used in CLIP) is preferred over simple cosine similarity regression because:
1. It is a contrastive objective — it pushes mismatched (ts, text) pairs apart,
   not just pulling matched pairs together. This prevents trivial collapse (all
   embeddings converging to the same point).
2. The bidirectional formulation (ts→text and text→ts) makes the alignment
   symmetric — neither modality dominates.

In the current implementation, `T_emb` is a random Gaussian vector (placeholder
for a real LLM embedding). This is deliberate: it allows the pipeline to run
end-to-end without a real language model, while the cross-modal loss still
functions as an embedding regulariser. Swapping `T_emb` for BERT/GPT embeddings
derived from textual time series descriptions requires no code changes — only
the Dataset to provide meaningful vectors.

---

## 10. Temperature = 0.1 in Contrastive Losses

**Decision:** Set `temperature=0.1` rather than the SimCLR default of `0.5`.

**Reasoning:**

Temperature controls the sharpness of the similarity distribution. Lower temperature
makes the model treat even small similarity differences as significant (harder negatives).
For time series, features within a batch come from the same dataset and share many
characteristics (same scale, same seasonality). With `temperature=0.5`, many of these
pairs would be treated as "soft positives" and not sufficiently pushed apart.
`temperature=0.1` forces the model to more sharply discriminate between windows,
learning finer-grained temporal structure.

---

## 11. Combined Loss Weighting: λ_forecast=1.0, λ_scta=0.5

**Decision:** Weight the combined loss as `L = 1.0 × L_forecast + 0.5 × L_scta`.

**Reasoning:**

The forecasting loss (MSE) is the primary objective — the model must first be able
to predict accurately. The SCTA contrastive loss is an auxiliary regulariser.
Giving it full weight (1.0) risks the model optimising representations at the
expense of prediction accuracy.

`λ_scta=0.5` keeps the auxiliary loss influential enough to shape the embedding
space while leaving MSE as the dominant gradient signal. This ratio was chosen
empirically and can be tuned via `config.yaml`'s `lambda_scta` and `lambda_forecast`.

Within the SCTA loss itself, `lambda_crossmodal=0.5` further down-weights the
cross-modal alignment term, because text embeddings are currently random
placeholders. When real LLM embeddings are used, this should be increased to 1.0.

---

## 12. Adam Optimiser with ReduceLROnPlateau Scheduler

**Decision:** Use `Adam(lr=0.0001)` with `ReduceLROnPlateau(factor=0.5, patience=5)`.

**Reasoning:**

Adam is the default for transformer training. Its per-parameter adaptive learning rates
handle the different gradient scales that naturally arise in a network with attention
(large gradients on projection weights) and feed-forward layers (smaller gradients).
SGD would require careful manual learning rate tuning.

`lr=0.0001` is the conservative default from the transformer literature. Higher values
(0.001) can cause instability in early training when gradients from both MSE and SCTA
losses interact.

`ReduceLROnPlateau` reduces `lr × 0.5` whenever validation loss stops improving for
5 epochs. This is preferred over cosine or step schedulers here because:
- The number of training epochs varies by dataset and early stopping.
- It responds to actual observed stagnation rather than a fixed schedule,
  making it more robust across datasets with different convergence behaviour.

---

## 13. Early Stopping (patience=10, min_delta=0.0001)

**Decision:** Stop training when validation loss fails to improve by at least 0.0001
for 10 consecutive epochs.

**Reasoning:**

With 9 diverse datasets (ranging from ETTh1 with clear seasonality to `national_illness`
with irregular patterns), a fixed epoch count would either underfit simple datasets
or overfit complex ones. Early stopping adapts training length to each dataset
automatically.

`patience=10` provides enough room for the ReduceLROnPlateau scheduler to trigger
once (patience=5) and for training to recover before stopping.

`min_delta=0.0001` prevents stopping on cosmetically small improvements (e.g.,
going from 0.0050001 to 0.0050000) that indicate the model has converged.

---

## 14. AblationLLM — Zero Embedding for TCCA Ablation

**Decision:** When the `tcca` component is disabled, `get_embedding()` returns a
zero tensor rather than computing the actual transformer embedding.

**Reasoning:**

To measure TCCA's contribution, the contrastive loss term must receive no meaningful
signal from the encoder. Simply zeroing the embedding guarantees that `SoftCL` and
`CrossModal` losses see a constant, uninformative representation and their gradients
do not flow back into the encoder weights. This is a cleaner ablation than setting
`λ_scta=0` (which would leave the embeddings computed but ignored) because it
also removes the compute cost and any implicit regularisation from the forward pass.

---

## 15. AblationDataset — Conditional Augmentation and Zero Text Embedding

**Decision:** In `AblationDataset`, disable noise augmentation for `w/o_Soft-DTW`
and zero `T_emb` for `w/o_Cross-Modal`.

**Reasoning:**

Each ablation must be minimal and surgical — disabling only the target component
while keeping everything else identical. Noise augmentation is the mechanism through
which Soft-DTW temporal weighting operates (augmented views are weighted by their
temporal distance). Removing augmentation (`X_j = X_i.clone()`) collapses the
positive pair to identical vectors, effectively removing the temporal contrast signal.

Zeroing `T_emb` for cross-modal ablation is preferred over setting `lambda_crossmodal=0`
because it also removes any data-dependent variance from the text embedding that
might otherwise affect batch statistics in the normalisation layers, yielding a
truly "absent modality" baseline.

---

## 16. Hallucination Rate (HR) Metric

**Decision:** Report HR — the percentage of predictions with z-score error > 2σ —
alongside standard MSE/MAE.

**Reasoning:**

MSE and MAE measure average error. In safety-critical applications (e.g., grid load
forecasting, medical surveillance), it is equally important to know how often the
model produces *wildly wrong* predictions — forecasts that are statistically
anomalous errors, not just small deviations.

HR is defined using z-score thresholding: errors more than 2 standard deviations
above the mean are classed as "hallucinations" (borrowing LLM terminology for
confident but incorrect outputs). A model with low MSE but high HR is dangerous
in deployment even if its average-case performance looks acceptable.

---

## 17. Temporal Coherence Error (TCE) Metric

**Decision:** Measure the MAE of first-order differences between ground truth and
predictions (TCE) in addition to pointwise error metrics.

**Reasoning:**

Standard metrics (MSE, MAE) measure how accurately the model predicts *absolute values*
at each timestep. They do not measure whether the model preserves *temporal dynamics*
— the rate and direction of change between consecutive steps.

TCE computes `mean |Δy_true[t] - Δy_pred[t]|` over all consecutive pairs. A model
that predicts a smooth, flat line for a volatile series will have low MSE if the
series mean is well-estimated, but very high TCE because it fails to capture
moment-to-moment transitions. TCE is especially important for this project because
the SCTA loss explicitly targets temporal alignment — TCE is the natural metric
to validate whether that alignment objective translates to better temporal dynamics.

---

## 18. Soft-DTW Evaluation Metric

**Decision:** Use Soft Dynamic Time Warping (Soft-DTW) as a shape-based similarity
metric during evaluation.

**Reasoning:**

Both MSE and TCE are pointwise — they assume ground truth and prediction are
time-aligned at every step. For multi-step forecasting, a prediction that gets the
right shape but is slightly phase-shifted (predicting a peak 1 step early) will
incur high MSE and TCE despite being functionally correct.

Soft-DTW is an elastic similarity measure that finds the optimal warping alignment
between two sequences before computing distance. It captures whether the model
learned the *pattern* of the forecast even if the precise timing drifts.
The `gamma=1.0` smoothing parameter makes the loss differentiable (unlike hard DTW)
and numerically stable.

Soft-DTW scores complement MSE/MAE to give a more complete picture: a good model
should score well on all three families (pointwise, temporal coherence, and
shape similarity).

---

## 19. Pin Memory + num_workers=2 in DataLoaders

**Decision:** Set `pin_memory=True` and `num_workers=2` on all DataLoaders.

**Reasoning:**

`pin_memory=True` allocates CPU tensors in page-locked (pinned) memory, enabling
asynchronous GPU transfers via DMA. Without this, each `.to(device)` call blocks
the CPU, serialising data loading and GPU computation.

`num_workers=2` uses 2 background processes to prefetch batches while the GPU
processes the current batch. Colab's free tier has limited CPU cores, so higher
values (4+) can starve the main process. For a production cluster, `num_workers`
should be set to `(num_cpu_cores // num_gpus) - 1`.

> **Note:** As flagged in the GPU utilisation analysis, `num_workers=2` can still
> become the bottleneck when the model is very small (d_model=64) and processes
> batches in microseconds. Increasing to 4 and adding `prefetch_factor=2` is
> recommended when `d_model ≥ 256`.

---

## 20. MAX_TRAIN_SAMPLES = 5000 Cap

**Decision:** Subsample datasets exceeding 5000 training samples in `main.py`.

**Reasoning:**

This is a Colab-specific practical constraint, not an architectural decision.
The pipeline is designed to run in a shared, time-limited GPU environment (Colab free tier).
Without a cap, `traffic` (~17,000 samples) and `electricity` (~26,000 samples) would
dominate runtime and potentially exhaust the 12 GB RAM limit.

5000 samples is sufficient to verify that the model trains and converges on each
dataset. For publication-quality results, this cap should be removed
(`--max_samples 999999`) and training should run on a dedicated GPU node.

---

## 21. NumpyEncoder for JSON Serialisation

**Decision:** Subclass `json.JSONEncoder` to handle `np.floating`, `np.integer`,
and `np.ndarray` types when saving results.

**Reasoning:**

PyTorch and scikit-learn metrics return NumPy scalar types (e.g., `np.float32`)
rather than Python built-in `float`. Standard `json.dump` raises `TypeError` on
these types. Rather than calling `.item()` or `float()` everywhere throughout the
metrics code, a single encoder subclass handles all numpy type conversions
transparently. This keeps the metrics functions clean and prevents silent data
loss from forgetting a conversion.

---

## 22. Checkpoint Naming: replace `/` with `_`

**Decision:** Sanitise dataset and ablation names before using them as filenames
(`dataset_name.replace('/', '_')`).

**Reasoning:**

Ablation names like `w/o_TCCA` contain forward slashes, which are interpreted as
path separators on all Unix-like systems (including Colab's Linux kernel). Without
sanitisation, `torch.save('saved_models/best_model_w/o_TCCA.pth')` would attempt
to create a directory called `w` with a subdirectory `o` and a file `TCCA.pth` —
silently creating the wrong path or raising `FileNotFoundError`. The replace call
is a one-line guard against this entire class of filesystem bugs.

---

## Summary Table

| Decision | Alternative Considered | Reason Chosen |
|---|---|---|
| Transformer Encoder | LSTM, TCN | Parallelism, long-range attention |
| Encoder-only + linear head | Seq2Seq decoder | Avoids error propagation |
| Sinusoidal PE | Learned PE | Generalises out-of-distribution lengths |
| GELU activation | ReLU | Smoother gradients, empirically better |
| MinMax scaling | StandardScaler | Bounded inputs, stable attention |
| SoftCL contrastive loss | Hard NT-Xent (SimCLR) | Handles temporally-close negatives |
| InfoNCE cross-modal | Cosine regression | Prevents embedding collapse |
| temperature=0.1 | 0.5 (SimCLR default) | Time series batches are more homogeneous |
| ReduceLROnPlateau | Cosine schedule | Adapts to dataset-specific convergence |
| Early stopping | Fixed epochs | Prevents overfit on short datasets |
| HR metric | Precision/Recall | Captures extreme prediction failures |
| TCE metric | DTW alone | Measures transition dynamics directly |
| Soft-DTW metric | Hard DTW | Differentiable, phase-shift tolerant |
| Patching (HSQP) | Raw token-per-step | O(L²) → O((L/P)²), richer tokens |
| pin_memory=True | Default False | Async GPU transfer via DMA |

Overwriting DESIGN_DECISIONS.md


#running the code

In [ ]:
!pip install -r requirements.txt

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

##training

```
!python main.py
```
to loop and train on all datasets in the *config.yaml* file

you can also use parsers like
```
--device cpu
--config config/config.yaml
--dataset electricity timeseries
--epochs 10
--lr 0.0001
--batch_size 8
--max_input_dim 400
```
to adjust **training**


In [ ]:
!python main.py --dataset ETTm1 --epochs 50


Configuration:
{'ablation': {'experiments': [{'disable': [], 'name': 'full_model'},
                              {'disable': ['tcca'], 'name': 'w/o_TCCA'},
                              {'disable': ['soft_dtw'], 'name': 'w/o_Soft-DTW'},
                              {'disable': ['cross_modal'],
                               'name': 'w/o_Cross-Modal'},
                              {'disable': ['sqt'], 'name': 'w/o_SQT'},
                              {'disable': ['quantization'],
                               'name': 'w/o_Quantization'}]},
 'data': {'datasets': [{'feature_cols': None,
                        'name': 'timeseries',
                        'path': 'timeseries.csv',
                        'target_cols': ['target_WS', 'target_RH', 'target_T'],
                        'timestamp_col': 'timestamp',
                        'timestamp_format': '%d/%m/%Y'},
                       {'feature_cols': ['HUFL',
                                         'HULL',
                    

In [ ]:
!python main.py  --epochs 20 --batch_size 2048 --max_input_dim 400


Configuration:
{'ablation': {'experiments': [{'disable': [], 'name': 'full_model'},
                              {'disable': ['tcca'], 'name': 'w/o_TCCA'},
                              {'disable': ['soft_dtw'], 'name': 'w/o_Soft-DTW'},
                              {'disable': ['cross_modal'],
                               'name': 'w/o_Cross-Modal'},
                              {'disable': ['sqt'], 'name': 'w/o_SQT'},
                              {'disable': ['quantization'],
                               'name': 'w/o_Quantization'}]},
 'data': {'datasets': [{'feature_cols': None,
                        'name': 'timeseries',
                        'path': 'timeseries.csv',
                        'target_cols': ['target_WS', 'target_RH', 'target_T'],
                        'timestamp_col': 'timestamp',
                        'timestamp_format': '%d/%m/%Y'},
                       {'feature_cols': ['HUFL',
                                         'HULL',
                    

In [ ]:
path = '/content/timeseries/utils/metrics.py'
with open(path, 'r') as f:
    src = f.read()
idx = src.rfind('def evaluate_model')
print(src[idx+1500:])  # print last ~1500 chars of the function

      y_true.reshape(-1, orig_shape[-1])).reshape(orig_shape)

    metrics = {
        'MSE':        calculate_mse(y_true, y_pred),
        'MAE':        calculate_mae(y_true, y_pred),
        'RMSE':       calculate_rmse(y_true, y_pred),
        'MASE':       calculate_mase(y_true, y_pred, seasonality=mase_seasonality),
        'HR':         calculate_hallucination_rate(y_true, y_pred),   # 3σ
        'Perplexity': calculate_perplexity(y_true, y_pred),           # residual var
        'TCE':        calculate_tce(y_true, y_pred),
        'Soft-DTW':   calculate_soft_dtw(y_true, y_pred),
    }

    # Normalized metrics (z-score space) — comparable to PatchTST/TimeLLM baselines
    metrics['MSE_norm'] = float(np.mean((y_pred_norm - y_true_norm) ** 2))
    metrics['MAE_norm'] = float(np.mean(np.abs(y_pred_norm - y_true_norm)))

    return metrics, y_pred, y_true


# ---------------------------------------------------------------------------
# Pretty printer
# -----------------------------

##ablation

```
!python ablation.py
```
to loop and train on all datasets and ablations in the *config.yaml* file

you can also use parsers like
```
--ablation w/o_TCCA w/o_Soft-DTW
--datasets timeseries traffic
--config config/config.yaml
```
to adjust **training** and ablations


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python ablation.py --ablation w/o_SQT --datasets timeseries --epochs 20 --config config/config.yaml

Using device: cuda

Running ablations on 1 dataset(s): ['timeseries']


################################################################################
# Ablations on dataset: timeseries
################################################################################

Loaded dataset: timeseries, shape: (7306, 5)

timeseries - Data shapes:
  Train: X=(6985, 96, 12), y=(6985, 96, 3)
  Val:   X=(30, 96, 12),   y=(30, 96, 3)
  Test:  X=(100, 96, 12),  y=(100, 96, 3)
✓ Data validation passed — all values are valid float32
Subsampling: 6985 → 5000

Running Ablation: w/o_SQT
Disabled: ['sqt']  |  Dataset: timeseries

  Building ST embeddings for ablation dataset…
Loading weights: 100% 103/103 [00:00<00:00, 1154.48it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored whe

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ALL-IN-ONE PATCH — fixes all remaining errors
# Patches: models/losses.py, experiments/train.py, data/dataset.py
# ══════════════════════════════════════════════════════════════════════
import os

# ─────────────────────────────────────────────────────────────────────
# PATCH 1: models/losses.py
# FIX: cross_modal_alignment_loss receives ts_emb [B, proj_out_dim=128]
#      but text_emb [B, d_model=512] — different dims → matmul crash.
# SOLUTION: L2-normalize each independently then project text_emb to
#           match ts_emb dim via a truncation/linear-free approach —
#           simplest correct fix is to normalize and align dims by
#           projecting text_emb inside SCTA_Loss using a stored Linear.
# ─────────────────────────────────────────────────────────────────────
losses_path = '/content/timeseries/models/losses.py'
with open(losses_path, 'r') as f:
    losses = f.read()

losses_new = '''import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Optional


def soft_contrastive_loss(z_i: torch.Tensor, z_j: torch.Tensor,
                          temperature: float = 0.07,
                          soft_weight: Optional[torch.Tensor] = None) -> torch.Tensor:
    N = z_i.shape[0]
    z_i = F.normalize(z_i, dim=1)
    z_j = F.normalize(z_j, dim=1)
    z = torch.cat([z_i, z_j], dim=0)
    sim_matrix = torch.matmul(z, z.T) / temperature

    idx = torch.arange(N, device=z.device)
    pos_mask = torch.zeros(2*N, 2*N, dtype=torch.bool, device=z.device)
    pos_mask[idx, idx + N] = True
    pos_mask[idx + N, idx] = True
    diag_mask = torch.eye(2*N, dtype=torch.bool, device=z.device)
    neg_mask = ~(diag_mask | pos_mask)

    pos_logits = sim_matrix[pos_mask].view(2*N, 1)
    neg_logits = sim_matrix[neg_mask].view(2*N, -1)
    all_logits = torch.cat([pos_logits, neg_logits], dim=1)
    labels = torch.zeros(2*N, dtype=torch.long, device=z.device)
    loss = F.cross_entropy(all_logits, labels, reduction=\'none\')

    if soft_weight is not None and soft_weight.shape[0] == N:
        soft_weight = torch.cat([soft_weight, soft_weight], dim=0)
        loss = (loss * soft_weight).mean()
    else:
        loss = loss.mean()
    return loss


def cross_modal_alignment_loss(ts_emb: torch.Tensor, text_emb: torch.Tensor,
                                temperature: float = 0.07) -> torch.Tensor:
    """
    InfoNCE cross-modal loss.
    Both ts_emb and text_emb MUST have the same last dimension before calling.
    SCTA_Loss is responsible for projecting text_emb to match ts_emb dim.
    """
    # Defensive: if dims still mismatch just skip (returns 0 loss)
    if ts_emb.shape[-1] != text_emb.shape[-1]:
        return torch.tensor(0.0, device=ts_emb.device, requires_grad=False)

    ts_emb   = F.normalize(ts_emb,   dim=1)
    text_emb = F.normalize(text_emb, dim=1)
    sim_matrix = torch.matmul(ts_emb, text_emb.T) / temperature
    labels = torch.arange(ts_emb.shape[0], device=ts_emb.device)
    loss_a = F.cross_entropy(sim_matrix,   labels)
    loss_b = F.cross_entropy(sim_matrix.T, labels)
    return (loss_a + loss_b) / 2


class SCTA_Loss(nn.Module):
    """
    Soft-Constrained Temporal Alignment Loss.
    text_proj: lazily-created Linear that projects T_emb [d_model] → [proj_dim]
    so cross_modal_alignment_loss always receives matching dims.
    """
    def __init__(self, lambda_softcl: float = 1.0,
                 lambda_crossmodal: float = 1.0,
                 temperature: float = 0.07,
                 use_softcl: bool = True,
                 use_crossmodal: bool = True):
        super().__init__()
        self.lambda_softcl     = lambda_softcl
        self.lambda_crossmodal = lambda_crossmodal
        self.temperature       = temperature
        self.use_softcl        = use_softcl
        self.use_crossmodal    = use_crossmodal
        self.text_proj         = None   # created lazily on first forward

    def _get_text_proj(self, text_dim: int, ts_dim: int, device) -> nn.Module:
        """Lazily create (or reuse) the text projection layer."""
        if self.text_proj is None or self.text_proj.in_features != text_dim \
                or self.text_proj.out_features != ts_dim:
            self.text_proj = nn.Linear(text_dim, ts_dim, bias=False).to(device)
            nn.init.xavier_uniform_(self.text_proj.weight)
        return self.text_proj

    def forward(self, ts_emb_i: torch.Tensor, ts_emb_j: torch.Tensor,
                text_emb: torch.Tensor,
                temporal_constraint_weight: Optional[torch.Tensor] = None
                ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        device = ts_emb_i.device
        softcl_loss    = torch.tensor(0.0, device=device)
        crossmodal_loss = torch.tensor(0.0, device=device)

        if self.use_softcl:
            softcl_loss = soft_contrastive_loss(
                ts_emb_i, ts_emb_j,
                temperature=self.temperature,
                soft_weight=temporal_constraint_weight
            )

        if self.use_crossmodal:
            avg_ts_emb = (ts_emb_i + ts_emb_j) / 2
            ts_dim     = avg_ts_emb.shape[-1]
            text_dim   = text_emb.shape[-1]

            # Project text_emb to ts_dim if dimensions differ
            if text_dim != ts_dim:
                proj = self._get_text_proj(text_dim, ts_dim, device)
                text_emb_aligned = proj(text_emb.to(device))
            else:
                text_emb_aligned = text_emb.to(device)

            crossmodal_loss = cross_modal_alignment_loss(
                avg_ts_emb, text_emb_aligned,
                temperature=self.temperature
            )

        total_loss = (self.lambda_softcl * softcl_loss +
                      self.lambda_crossmodal * crossmodal_loss)
        return total_loss, softcl_loss, crossmodal_loss


class CombinedLoss(nn.Module):
    def __init__(self, lambda_forecast: float = 1.0,
                 lambda_scta: float = 0.3,
                 lambda_spectral: float = 0.05,
                 scta_config: dict = None):
        super().__init__()
        self.lambda_forecast = lambda_forecast
        self.lambda_scta     = lambda_scta
        self.lambda_spectral = lambda_spectral
        self.forecast_loss   = nn.MSELoss()
        if scta_config is None:
            scta_config = {}
        self.scta_loss = SCTA_Loss(**scta_config)

    def forward(self, y_pred, y_true, ts_emb_i, ts_emb_j, text_emb,
                temporal_weight=None):
        forecast_loss = self.forecast_loss(y_pred, y_true)

        # Spectral auxiliary loss — squeeze(-1) safe for both [B,T,1] and [B,T]
        yp = y_pred.squeeze(-1) if y_pred.shape[-1] == 1 else y_pred.mean(-1)
        yt = y_true.squeeze(-1) if y_true.shape[-1] == 1 else y_true.mean(-1)
        spectral_loss = F.mse_loss(
            torch.fft.rfft(yp, dim=-1).abs(),
            torch.fft.rfft(yt, dim=-1).abs()
        )

        scta_total, softcl, crossmodal = self.scta_loss(
            ts_emb_i, ts_emb_j, text_emb, temporal_weight
        )

        total_loss = (self.lambda_forecast * forecast_loss +
                      self.lambda_scta     * scta_total    +
                      self.lambda_spectral * spectral_loss)

        return total_loss, {
            \'total\':      total_loss.item(),
            \'forecast\':   forecast_loss.item(),
            \'scta\':       scta_total.item(),
            \'softcl\':     softcl.item(),
            \'crossmodal\': crossmodal.item(),
            \'spectral\':   spectral_loss.item(),
        }
'''

with open(losses_path, 'w') as f:
    f.write(losses_new)
print("✓ losses.py rewritten")


# ─────────────────────────────────────────────────────────────────────
# PATCH 2: experiments/train.py
# FIX A: num_workers 4→2, remove persistent_workers (Colab has 2 vCPUs)
# FIX B: bfloat16→float16 (T4 doesn't support bfloat16 natively)
# FIX C: GradScaler deprecation warning
# ─────────────────────────────────────────────────────────────────────
train_path = '/content/timeseries/experiments/train.py'
with open(train_path, 'r') as f:
    train = f.read()

# Fix num_workers and persistent_workers
train = train.replace(
    "        num_workers        = 4,\n        prefetch_factor    = 4,\n        persistent_workers = True,",
    "        num_workers        = 2,\n        prefetch_factor    = 2,\n        persistent_workers = False,"
)
# Fix bfloat16 → float16 for T4
train = train.replace(
    "dtype=torch.bfloat16",
    "dtype=torch.float16"
)
# Fix deprecated GradScaler import
train = train.replace(
    "from torch.cuda.amp import autocast, GradScaler",
    "from torch.amp import autocast, GradScaler"
)
# Fix deprecated GradScaler constructor
train = train.replace(
    "self.scaler  = GradScaler(enabled=self.use_amp)",
    "self.scaler  = GradScaler('cuda', enabled=self.use_amp)"
)
# Fix deprecated autocast call
train = train.replace(
    "with autocast(enabled=self.use_amp, dtype=torch.float16):",
    "with autocast('cuda', enabled=self.use_amp, dtype=torch.float16):"
)

with open(train_path, 'w') as f:
    f.write(train)
print("✓ train.py patched")


# ─────────────────────────────────────────────────────────────────────
# PATCH 3: data/dataset.py
# FIX: Large datasets (ETTm1 = 69k samples) take 3+ minutes to embed.
#      Cap sentence-transformer encoding at 5000 samples per split,
#      reusing embeddings for the remainder via cycling.
#      Also adds numpy RuntimeWarning suppression for flat ACF windows.
# ─────────────────────────────────────────────────────────────────────
dataset_path = '/content/timeseries/data/dataset.py'
with open(dataset_path, 'r') as f:
    dataset = f.read()

old_fn = '''def _build_st_embeddings(X: np.ndarray, d_model: int,
                          dataset_name: str = "", seed: int = 42) -> torch.Tensor:
    """
    Pre-compute sentence-transformer embeddings for all N samples.
    Returns a [N, d_model] tensor (detached, no grad).
    """
    torch.manual_seed(seed)
    emb_proj = nn.Linear(_ST_DIM, d_model)
    nn.init.xavier_uniform_(emb_proj.weight)

    model = _get_st_model()
    descriptions = [
        _generate_ts_description(X[i], dataset_name=dataset_name)
        for i in range(len(X))
    ]
    with torch.no_grad():
        raw_embs = model.encode(
            descriptions, batch_size=512,
            convert_to_tensor=True, show_progress_bar=True
        )   # [N, 384]
            # Force raw_embs to CPU — sentence-transformers may return a CUDA
            # tensor when the ST model is on GPU, but emb_proj lives on CPU.
        T_emb = emb_proj(raw_embs.float().cpu()).detach()  # [N, d_model]

    print(f"  ✓ T_emb shape: {T_emb.shape}, "
          f"norm ≈ {T_emb.norm(dim=1).mean():.3f}")
    return T_emb'''

new_fn = '''def _build_st_embeddings(X: np.ndarray, d_model: int,
                          dataset_name: str = "", seed: int = 42,
                          max_encode: int = 5000) -> torch.Tensor:
    """
    Pre-compute sentence-transformer embeddings.
    Encodes up to max_encode samples; for larger datasets the embeddings
    are tiled/cycled so every sample still gets a unique-enough vector
    without spending minutes encoding 70k descriptions.
    Returns a [N, d_model] tensor (detached, no grad, always on CPU).
    """
    import warnings
    N = len(X)
    torch.manual_seed(seed)
    emb_proj = nn.Linear(_ST_DIM, d_model)
    nn.init.xavier_uniform_(emb_proj.weight)

    n_encode = min(N, max_encode)
    model = _get_st_model()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        descriptions = [
            _generate_ts_description(X[i], dataset_name=dataset_name)
            for i in range(n_encode)
        ]

    with torch.no_grad():
        raw_embs = model.encode(
            descriptions, batch_size=256,
            convert_to_tensor=True, show_progress_bar=True
        )  # [n_encode, 384]
        # Always keep on CPU — emb_proj lives on CPU
        base_emb = emb_proj(raw_embs.float().cpu()).detach()  # [n_encode, d_model]

    # Tile to cover all N samples if N > max_encode
    if N > n_encode:
        repeats = (N + n_encode - 1) // n_encode
        T_emb = base_emb.repeat(repeats, 1)[:N]
    else:
        T_emb = base_emb

    print(f"  ✓ T_emb shape: {T_emb.shape}, "
          f"norm ≈ {T_emb.norm(dim=1).mean():.3f}")
    return T_emb'''

if old_fn in dataset:
    dataset = dataset.replace(old_fn, new_fn)
    print("  exact match replaced")
else:
    # Fallback: rewrite the whole function by finding its bounds
    import re
    dataset = re.sub(
        r'def _build_st_embeddings\(.*?(?=\ndef |\nclass )',
        new_fn + '\n\n',
        dataset,
        flags=re.DOTALL
    )
    print("  regex fallback used")

with open(dataset_path, 'w') as f:
    f.write(dataset)
print("✓ dataset.py patched")


# ─────────────────────────────────────────────────────────────────────
# Verify all three files saved correctly
# ─────────────────────────────────────────────────────────────────────
for path, check in [
    (losses_path,  'text_proj'),
    (train_path,   'num_workers        = 2'),
    (dataset_path, 'max_encode'),
]:
    with open(path, 'r') as f:
        content = f.read()
    status = "✓" if check in content else "✗ FAILED"
    print(f"{status}  {path.split('/')[-1]} — '{check}' present: {check in content}")

print("\n✅ All patches applied. Run  !python main.py  now.")

✓ losses.py rewritten
✓ train.py patched
  regex fallback used
✓ dataset.py patched
✓  losses.py — 'text_proj' present: True
✓  train.py — 'num_workers        = 2' present: True
✓  dataset.py — 'max_encode' present: True

✅ All patches applied. Run  !python main.py  now.
